# Digikala AI Shopping Assistant — Standalone Notebook
### QBC13 · AI · Group 6 · Project 3

A single, self-contained file that builds and demonstrates the **whole project**:

1. **Data** — clean the raw Digikala data into a documented schema + Plotly EDA
2. **Assistant** — Persian discovery · review Q&A · comparison · manager analytics
   (hybrid dense+BM25 retrieval, grounded citations, extractive $0 fallback)
3. **Prediction** — classify `recommendation_status` (Macro-F1)
4. **Evaluation** — retrieval / grounding / latency / cost / failure analysis

Everything runs top-to-bottom on a small **reproducible sample** by default so it
is easy to *Run All*; set `SAMPLE_SIZE = None` to use more data. The production
package (`src/digikala/` + `run.py`) runs the same code on the full corpus.

> Each section first shows the library code (identical to the package), then runs it.

## 0 · Setup & configuration
Shared imports and a small config object (paths, sample size, run mode). API keys come from a local `.env` file (see `.env.sample` in the repo root) via python-dotenv -- never pasted into the notebook.

In [1]:
# If needed:  pip install pandas numpy pyarrow scikit-learn plotly hazm persiantools \
#                         sentence-transformers torch huggingface-hub scipy python-dotenv
import os, re, math, json, time, ast, logging, shutil, unicodedata, types
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
logging.basicConfig(level=logging.WARNING)

try:
    from dotenv import load_dotenv
    load_dotenv()                      # populates os.environ from a local .env, if present
except ImportError:
    pass

# ---- knobs -------------------------------------------------------------
SAMPLE_SIZE = 20000          # comments to sample for the demo; set None for "all"
JUDGE_MODE  = "none"         # none | local | free | paid | hosted_auto  (Phase 4 LLM-as-judge)

# RUN_MODE: explicit env var wins; otherwise auto-detect a hosted key from
# .env (hosted_auto tries Groq, then the paid gateway); otherwise $0 local.
_explicit_mode = os.environ.get("DIGIKALA_RUN_MODE")
if _explicit_mode:
    RUN_MODE = _explicit_mode
elif os.environ.get("GROQ_API_KEY") or os.environ.get("PAID_API_KEY"):
    RUN_MODE = "hosted_auto"
else:
    RUN_MODE = "extractive"    # $0, always-grounded default for the demo notebook

# ---- paths (found whether you run from the repo root or the notebooks/ folder) --
def _find_raw():
    # walk up from the cwd looking for the real CSVs; else default (download_raw fills it)
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "data/raw/digikala-products.csv").exists():
            return base / "data/raw"
    p = Path.cwd() / "data/raw"; p.mkdir(parents=True, exist_ok=True); return p
RAW  = _find_raw()
BASE = Path("nb_artifacts"); BASE.mkdir(exist_ok=True)

GROQ_MODEL_PRICES = {"openai/gpt-oss-20b": (0.075, 0.30), "openai/gpt-oss-120b": (0.15, 0.60),
                     "llama-3.3-70b-versatile": (0.59, 0.79)}
_groq_model = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")

config = types.SimpleNamespace(
    HF_REPO_ID="RadeAI/Digikala_comments_products",
    HF_REVISION="89c3133b169c8d3793db8834f56f32fee33d9db0",
    PRODUCTS_CSV=RAW/"digikala-products.csv", COMMENTS_CSV=RAW/"digikala-comments.csv",
    PRODUCTS_CLEAN=BASE/"products_clean.parquet", COMMENTS_CLEAN=BASE/"comments_clean.parquet",
    PHASE1_REPORT=BASE/"phase1_report.json", PRODUCT_INDEX_DIR=BASE/"index",
    PROCESSED_DIR=BASE, FIGURES_DIR=BASE, METRICS_DIR=BASE, MODELS_DIR=BASE, RAW_DIR=RAW,
    CHUNK_SIZE=200_000, COMMENTS_SAMPLE_SIZE=SAMPLE_SIZE, RANDOM_SEED=42,
    RECOMMENDATION_CLASSES=("recommended","not_recommended","no_idea"), TOMAN_TO_RIAL=10,
    EMBEDDING_MODEL="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    EMBEDDING_DEVICE="cuda", TOP_K=8,
    RRF_CANDIDATE_POOL=200, REVIEW_CANDIDATE_POOL=80,
    PRODUCT_RRF_DENSE_WEIGHT=0.65, PRODUCT_RRF_SPARSE_WEIGHT=1.35,
    REVIEW_NEGATIVE_INTENT_WEIGHT=0.32, REVIEW_POSITIVE_INTENT_WEIGHT=0.28,
    RUN_MODE=RUN_MODE, LOCAL_BACKEND="transformers",
    HF_LLM_MODEL="Qwen/Qwen2.5-1.5B-Instruct", OLLAMA_BASE_URL="http://localhost:11434",
    OLLAMA_MODEL="qwen2.5:7b-instruct",
    GROQ_MODEL_PRICES=GROQ_MODEL_PRICES,
    PROVIDERS={
        "groq": {"base_url": "https://api.groq.com/openai/v1", "model": _groq_model,
                 "price_per_m": GROQ_MODEL_PRICES.get(_groq_model, (0.075, 0.30)),
                 "price_note": "Estimated list cost from configured per-million-token rates, not a Groq invoice.",
                 "billed": os.environ.get("DIGIKALA_GROQ_BILLED", "0").strip().lower() in {"1", "true", "yes"},
                 "api_key_env": "GROQ_API_KEY"},
        "openrouter": {"base_url": "https://openrouter.ai/api/v1", "model": "meta-llama/llama-3.3-70b-instruct:free",
                       "price_per_m": (0.0, 0.0), "price_note": "OpenRouter's :free tier.", "billed": False,
                       "api_key_env": "OPENROUTER_API_KEY"},
        "paid": {"base_url": os.environ.get("PAID_BASE_URL", "https://api.openai.com/v1"),
                 "model": os.environ.get("PAID_MODEL", "gpt-4o-mini"), "price_per_m": (0.15, 0.60),
                 "price_note": "Estimated list cost from configured per-million-token rates.",
                 "billed": True, "api_key_env": "PAID_API_KEY"},
    },
    FREE_PROVIDER="groq", HOSTED_PROVIDER_ORDER=("groq", "paid"),
    API_MAX_ATTEMPTS=3, API_RETRY_BASE_S=1.5, API_CONNECT_TIMEOUT_S=30, API_READ_TIMEOUT_S=120,
    LLM_MAX_NEW_TOKENS=512, LLM_TEMPERATURE=0.0,
    BUDGET_USD=5.0, BUDGET_LOG=BASE/"budget_log.jsonl", PAID_PRICE_PER_M=(0.15,0.60),
    JUDGE_MODE=JUDGE_MODE,
)
(config.PRODUCT_INDEX_DIR).mkdir(parents=True, exist_ok=True)
print("config ready | SAMPLE_SIZE =", SAMPLE_SIZE, "| RUN_MODE =", RUN_MODE,
      "| hosted key detected =", bool(os.environ.get("GROQ_API_KEY") or os.environ.get("PAID_API_KEY")))

config ready | SAMPLE_SIZE = 20000 | RUN_MODE = extractive | hosted key detected = False


## 1 · Phase 1 — Data cleaning & EDA
### 1.1 Persian text utilities

In [2]:
import re
import unicodedata

# hazm gives a better normalizer, but it's a heavy optional dep; degrade gracefully.
try:
    from hazm import Normalizer as _HazmNormalizer
    _hazm = _HazmNormalizer()
except Exception:                                   # pragma: no cover
    _hazm = None

# Persian/Arabic digits -> ASCII, and unify the Arabic ی/ک variants.
_DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
_CHAR_MAP = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک"})

_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]", re.UNICODE)
_INVISIBLE_RE = re.compile("[​-‏‪-‮⁦-⁩﻿]")
_WS_RE = re.compile(r"\s+")
_TOKEN_RE = re.compile(r"[^\W_]+", re.UNICODE)


def normalize(text: object, *, fold_digits: bool = True, drop_emoji: bool = False) -> str:
    """Normalize a Persian string. Returns '' for NaN/None."""
    if text is None or (isinstance(text, float) and text != text):
        return ""
    s = unicodedata.normalize("NFKC", str(text)).translate(_CHAR_MAP)
    s = _URL_RE.sub(" ", s)
    if drop_emoji:
        s = _EMOJI_RE.sub(" ", s)
    s = _hazm.normalize(s) if _hazm is not None else s
    if fold_digits:
        s = s.translate(_DIGIT_MAP)
    s = _INVISIBLE_RE.sub(" ", s)
    return _WS_RE.sub(" ", s).strip()


def tokenize(text: object) -> list[str]:
    return _TOKEN_RE.findall(normalize(text))


def tokenize_norm(text: object) -> list[str]:
    """Fast tokenizer for text that is ALREADY normalized (skips the hazm pass).
    Use for corpus building where inputs are the stored *_norm columns."""
    if text is None or (isinstance(text, float) and text != text):
        return []
    return _TOKEN_RE.findall(str(text))


def is_meaningful(text: object, min_chars: int = 2) -> bool:
    return len(normalize(text).replace("‌", "").strip()) >= min_chars


# ---- price constraint parser (used by the router) -----------------------
_NUM = r"(\d+(?:[./]\d+)?)"
_UNIT = r"(هزار|میلیون|میلیارد|k|K)?"
_CUR = r"(تومان|تومن|تومنی|تومانی|ریال)?"
_UPPER_WORDS = ("زیر", "کمتر", "حداکثر", "سقف", "تا", "ارزان‌تر", "پایین‌تر")
_LOWER_WORDS = ("بالای", "بیشتر", "بالاتر", "حداقل", "کف", "گران‌تر")
_DIRECTIONS = _UPPER_WORDS + _LOWER_WORDS
_PRICE_RE = re.compile(
    rf"(?:(?P<kw>{'|'.join(map(re.escape, _DIRECTIONS))})\s*)?"
    rf"(?P<num>{_NUM})\s*(?P<unit>{_UNIT})\s*{_CUR}")


def _unit_mult(u: str | None) -> float:
    return {"هزار": 1e3, "k": 1e3, "K": 1e3, "میلیون": 1e6, "میلیارد": 1e9}.get(u, 1.0)


def extract_price_constraint(text: object, toman_to_rial: int = 10) -> dict:
    """Pull {price_min, price_max} (in Rials) from a Persian query.

    A bare number with no direction word ("زیر"/"بالای"/...) is ignored, and the
    amount is treated as Tomans unless the user wrote "ریال" explicitly.
    """
    t = normalize(text)
    out: dict = {}
    for m in _PRICE_RE.finditer(t):
        kw = (m.group("kw") or "").strip()
        if not kw:
            continue
        val = float(m.group("num").replace("/", ".")) * _unit_mult(m.group("unit"))
        if (m.group(4) or "") != "ریال":            # group 4 is the currency
            val *= toman_to_rial
        if kw in _UPPER_WORDS:
            out["price_max"] = min(out.get("price_max", val), val)
        else:
            out["price_min"] = max(out.get("price_min", val), val)
    return out


def format_toman(value: object) -> str:
    """Rials -> a human 'Toman' string, or 'نامشخص' when missing."""
    if value is None or (isinstance(value, float) and value != value):
        return "نامشخص"
    try:
        return f"{int(float(value)) // 10:,}"
    except (TypeError, ValueError):
        return "نامشخص"

pt = types.SimpleNamespace(normalize=normalize, tokenize=tokenize, tokenize_norm=tokenize_norm,
    is_meaningful=is_meaningful, extract_price_constraint=extract_price_constraint,
    format_toman=format_toman, _DIGIT_MAP=_DIGIT_MAP)

### 1.2 Data IO — download + streaming/sampling

In [3]:
import logging
import os
import shutil

import numpy as np
import pandas as pd


log = logging.getLogger("digikala.dataio")

# The HF "Xet" transfer backend stalls on the big comments file; force plain HTTP.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")


def download_raw() -> None:
    """Fetch both CSVs from the pinned HF revision into data/raw if missing."""
    from huggingface_hub import hf_hub_download

    targets = {
        "digikala-products.csv": config.PRODUCTS_CSV,
        "digikala-comments.csv": config.COMMENTS_CSV,
    }
    for filename, dest in targets.items():
        if dest.exists() and dest.stat().st_size > 0:
            log.info("%s already present, skipping", dest.name)
            continue
        log.info("downloading %s @ %s", filename, config.HF_REVISION[:8])
        cached = hf_hub_download(repo_id=config.HF_REPO_ID, repo_type="dataset",
                                 filename=filename, revision=config.HF_REVISION)
        shutil.copy2(cached, dest)                   # copy out of the cache to keep the repo self-contained
        log.info("saved %s (%.0f MB)", dest.name, dest.stat().st_size / 1024**2)


def load_products() -> pd.DataFrame:
    """Products (~1.28M rows) fit in memory, so read the whole file."""
    return pd.read_csv(config.PRODUCTS_CSV, low_memory=False)


def iter_comment_chunks(chunksize: int | None = None):
    """Yield the comments CSV in row chunks so the cleaner never holds it all."""
    chunksize = chunksize or config.CHUNK_SIZE
    yield from pd.read_csv(config.COMMENTS_CSV, chunksize=chunksize, low_memory=False)


def load_comments(sample_size: int | None = "default") -> pd.DataFrame:
    """Read comments for the notebook: a reservoir sample by default, all if None."""
    if sample_size == "default":
        sample_size = config.COMMENTS_SAMPLE_SIZE
    if not sample_size:
        return pd.read_csv(config.COMMENTS_CSV, low_memory=False)
    return reservoir_sample(config.COMMENTS_CSV, sample_size, config.RANDOM_SEED)


def reservoir_sample(path, k: int, seed: int) -> pd.DataFrame:
    """Exact uniform k-row sample in one streaming pass, vectorized (fast even
    over 6M rows): give every row an i.i.d. random priority and keep the k
    smallest seen so far. Equivalent guarantee to classic reservoir sampling
    (every row equally likely to end up in the sample) but chunk-vectorized
    instead of a per-row Python loop, so it doesn't bottleneck a 6M-row scan."""
    if k <= 0:
        raise ValueError("sample size must be positive")
    rng = np.random.default_rng(seed)
    best = None
    for chunk in pd.read_csv(path, chunksize=config.CHUNK_SIZE, low_memory=False):
        chunk = chunk.copy()
        chunk["_sample_priority"] = rng.random(len(chunk))
        best = chunk if best is None else pd.concat([best, chunk], ignore_index=True)
        best = best.nsmallest(min(k, len(best)), "_sample_priority")
    if best is None:
        return pd.DataFrame()
    return (best.drop(columns="_sample_priority")
                .sample(frac=1, random_state=seed)     # shuffle away the priority order
                .reset_index(drop=True))


priority_sample = reservoir_sample                      # alias (same algorithm)

dataio = types.SimpleNamespace(download_raw=download_raw, load_products=load_products,
    iter_comment_chunks=iter_comment_chunks, load_comments=load_comments,
    reservoir_sample=reservoir_sample)

### 1.3 Cleaning — documented `_norm`/`_clean` schema

In [4]:
import ast
import json
import logging
import re

import numpy as np
import pandas as pd
from persiantools.jdatetime import JalaliDate


log = logging.getLogger("digikala.phase1")

_JALALI_MONTHS = {
    "فروردین": 1, "اردیبهشت": 2, "خرداد": 3, "تیر": 4, "مرداد": 5, "شهریور": 6,
    "مهر": 7, "آبان": 8, "آذر": 9, "دی": 10, "بهمن": 11, "اسفند": 12,
}
_TRUE = {"1", "true", "yes", "بله", "t", "y"}
_FALSE = {"0", "false", "no", "خیر", "f", "n"}
_GENERIC = "نامشخص"


# ---- small typed converters --------------------------------------------
def _to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def _to_bool(s: pd.Series) -> pd.Series:
    def conv(v):
        if pd.isna(v):
            return pd.NA
        t = str(v).strip().lower()
        return True if t in _TRUE else False if t in _FALSE else pd.NA
    return s.map(conv).astype("boolean")


def _parse_list_field(v) -> str:
    """advantages/disadvantages arrive as list-literal strings, e.g.
    "['جنسش خوبه\\r', 'خوش رنگه']" — pull the items out and join them."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).replace("\\r", " ").replace("\r", " ")
    try:
        parsed = ast.literal_eval(s)
        items = [str(x) for x in parsed] if isinstance(parsed, (list, tuple)) else [str(parsed)]
    except Exception:
        items = re.split(r"',\s*'", s.strip("[]"))
    items = [pt.normalize(it.strip(" '\"")) for it in items]
    return " ، ".join(it for it in items if it)


def _dedup(df: pd.DataFrame, report: dict, id_col: str) -> pd.DataFrame:
    n = len(df)
    df = df.drop_duplicates()
    report["dropped_exact_duplicates"] = n - len(df)
    if id_col in df.columns:
        n = len(df)
        df = df.drop_duplicates(subset=id_col, keep="first")
        report["dropped_duplicate_ids"] = n - len(df)
    return df


# ---- products -----------------------------------------------------------
def clean_products(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    report: dict = {"input_rows": len(df)}
    df = _dedup(df.copy(), report, id_col="id")
    out = pd.DataFrame()

    out["product_id"] = _to_num(df.get("id")).astype("Int64")
    out["title_fa"] = df.get("title_fa", "").map(pt.normalize)
    out["title_norm"] = out["title_fa"]

    for src, dst in [("Brand", "brand_norm"), ("Category1", "category1_norm"),
                     ("Category2", "category2_norm"), ("sub_category", "sub_category_norm"),
                     ("Seller", "seller_norm")]:
        out[dst] = df.get(src, pd.Series(index=df.index)).fillna(_GENERIC).map(pt.normalize)

    out["price_clean"] = _to_num(df.get("Price"))
    out["min_price_last_month"] = _to_num(df.get("min_price_last_month"))
    out["product_rate_clean"] = _to_num(df.get("Rate"))          # 0..100 scale in this dataset
    out["rate_count"] = _to_num(df.get("Rate_cnt")).fillna(0).astype("Int64")
    out["is_fake"] = _to_bool(df.get("Is_Fake", pd.Series(index=df.index)))

    # price 0 / missing means "not for sale", not a real price — flag it, keep the row
    report["zero_or_missing_price"] = int((out["price_clean"].isna() | (out["price_clean"] == 0)).sum())
    out["price_available"] = out["price_clean"].notna() & (out["price_clean"] > 0)

    # embedding text: title + categories + brand, skipping the generic placeholder
    def _ptext(r):
        parts = [r["title_norm"], r["category1_norm"], r["category2_norm"],
                 r["sub_category_norm"], r["brand_norm"]]
        return " ".join(p for p in parts if p and p != _GENERIC)
    out["product_text_norm"] = out.apply(_ptext, axis=1)

    out = out.dropna(subset=["product_id"]).reset_index(drop=True)
    report["output_rows"] = len(out)
    return out, report


# ---- comments -----------------------------------------------------------
def clean_comments(df: pd.DataFrame, valid_product_ids: set | None = None) -> tuple[pd.DataFrame, dict]:
    """Clean one comments dataframe (a chunk or the whole notebook sample)."""
    report: dict = {"input_rows": len(df)}
    df = _dedup(df.copy(), report, id_col="id")
    out = pd.DataFrame()

    def col(name):     # always a Series aligned to df, even if the column is absent
        return df[name] if name in df.columns else pd.Series(index=df.index, dtype="object")

    out["comment_id"] = _to_num(col("id")).astype("Int64")
    out["product_id"] = _to_num(col("product_id")).astype("Int64")
    out["title_norm"] = col("title").map(pt.normalize)
    out["body_norm"] = col("body").map(pt.normalize)
    out["advantages_norm"] = col("advantages").map(_parse_list_field)
    out["disadvantages_norm"] = col("disadvantages").map(_parse_list_field)

    # combined text used by retrieval and the classifier
    out["comment_text_norm"] = (
        out[["title_norm", "body_norm", "advantages_norm", "disadvantages_norm"]]
        .agg(" ".join, axis=1).map(pt.normalize))
    out["has_text"] = out["comment_text_norm"].map(pt.is_meaningful)

    out["rate_clean"] = _to_num(col("rate"))
    out["likes"] = _to_num(col("likes")).fillna(0).astype("Int64")
    out["dislikes"] = _to_num(col("dislikes")).fillna(0).astype("Int64")
    out["is_buyer"] = _to_bool(col("is_buyer"))
    out["true_to_size_rate"] = _to_num(col("true_to_size_rate"))

    if "created_at" in df.columns:
        out["created_at"] = _parse_datetime(df["created_at"])

    # recommendation label: validate against the 3 allowed classes
    rs = df.get("recommendation_status", pd.Series(index=df.index)).astype("string").str.strip().str.lower()
    out["recommendation_status"] = rs.where(rs.isin(config.RECOMMENDATION_CLASSES))
    out["recommendation_valid"] = out["recommendation_status"].isin(config.RECOMMENDATION_CLASSES)

    # does the comment point at a real product? (needed for RAG grounding + joins)
    if valid_product_ids is not None:
        out["has_product_match"] = out["product_id"].isin(valid_product_ids)
    else:
        out["has_product_match"] = out["product_id"].notna()

    report["rows_without_text"] = int((~out["has_text"]).sum())
    report["invalid_recommendation_labels"] = int((~out["recommendation_valid"]).sum())
    report["rows_without_product_match"] = int((~out["has_product_match"]).sum())
    out = out.dropna(subset=["comment_id"]).reset_index(drop=True)

    # Pin dtypes so every streamed chunk produces an identical parquet schema
    # (otherwise an all-null column in one chunk can mismatch a later one).
    if "created_at" not in out:
        out["created_at"] = pd.NaT
    out = out.astype({
        "comment_id": "Int64", "product_id": "Int64", "likes": "Int64", "dislikes": "Int64",
        "rate_clean": "float64", "true_to_size_rate": "float64",
        "title_norm": "string", "body_norm": "string", "advantages_norm": "string",
        "disadvantages_norm": "string", "comment_text_norm": "string",
        "recommendation_status": "string", "has_text": "boolean",
        "recommendation_valid": "boolean", "has_product_match": "boolean",
    })
    out["is_buyer"] = out["is_buyer"].astype("boolean")
    out["created_at"] = pd.to_datetime(out["created_at"], errors="coerce")
    report["output_rows"] = len(out)
    return out, report


def _parse_datetime(s: pd.Series) -> pd.Series:
    """created_at is a Jalali string like "23 شهریور 1402"; convert to Gregorian.
    Cached per distinct string since there are relatively few unique dates."""
    cache: dict[str, pd.Timestamp] = {}

    def parse_one(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return pd.NaT
        key = str(v).strip()
        if key in cache:
            return cache[key]
        result = pd.NaT
        parts = key.split()
        if len(parts) == 3 and parts[1] in _JALALI_MONTHS:
            try:
                day = int(parts[0].translate(pt._DIGIT_MAP))
                year = int(parts[2].translate(pt._DIGIT_MAP))
                result = pd.Timestamp(JalaliDate(year, _JALALI_MONTHS[parts[1]], day).to_gregorian())
            except Exception:
                result = pd.NaT
        cache[key] = result
        return result
    return s.map(parse_one)


# ---- driver -------------------------------------------------------------
def _merge_reports(total: dict, chunk: dict) -> None:
    for k, v in chunk.items():
        if isinstance(v, (int, float)):
            total[k] = total.get(k, 0) + v


def build(full: bool = True) -> dict:
    """Clean products (in memory) then stream-clean comments to Parquet.

    full=True streams the entire comments CSV chunk-by-chunk (run.py default).
    full=False cleans only the notebook sample from config.COMMENTS_SAMPLE_SIZE.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    dataio.download_raw()

    log.info("cleaning products")
    products, p_rep = clean_products(dataio.load_products())
    products.to_parquet(config.PRODUCTS_CLEAN, index=False)
    valid_ids = set(int(x) for x in products["product_id"].dropna())
    log.info("saved %s (%d rows)", config.PRODUCTS_CLEAN.name, len(products))

    c_total: dict = {}
    writer = None
    if full:
        log.info("streaming + cleaning comments in chunks of %d", config.CHUNK_SIZE)
        for i, chunk in enumerate(dataio.iter_comment_chunks()):
            cleaned, rep = clean_comments(chunk, valid_ids)
            _merge_reports(c_total, rep)
            table = pa.Table.from_pandas(cleaned, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(config.COMMENTS_CLEAN, table.schema)
            writer.write_table(table)
            if i % 5 == 0:
                log.info("  chunk %d done (%d rows cumulative)", i, c_total.get("output_rows", 0))
        if writer is not None:
            writer.close()
    else:
        comments, c_total = clean_comments(dataio.load_comments(), valid_ids)
        comments.to_parquet(config.COMMENTS_CLEAN, index=False)
    log.info("saved %s (%d rows)", config.COMMENTS_CLEAN.name, c_total.get("output_rows", 0))

    report = {"products": p_rep, "comments": c_total, "full": full}
    config.PHASE1_REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

### 1.4 Run Phase 1 (on the sample)
We draw an UNBIASED uniform sample of comments across the whole CSV (`dataio.reservoir_sample` -- not just the first N rows, which would bias toward file order), keep only the products they reference (so Q&A/managerial have matched reviews), and clean both. This is the exact same call `digikala.demo.sample_frames` makes, so the notebook and `python run.py demo` draw an identical sample. The full pipeline (`run.py clean`) streams all 6M comments.

In [5]:
# 1) unbiased sample of comments (same dataio.reservoir_sample call demo.py uses)
if not config.COMMENTS_CSV.exists():
    dataio.download_raw()
com_raw = dataio.reservoir_sample(config.COMMENTS_CSV, SAMPLE_SIZE or 20000, config.RANDOM_SEED)
wanted = set(pd.to_numeric(com_raw["product_id"], errors="coerce").dropna().astype(int))

# 2) load only the referenced products (chunked filter over the products CSV)
keep = []
for ch in pd.read_csv(config.PRODUCTS_CSV, chunksize=200_000, low_memory=False):
    ids = pd.to_numeric(ch["id"], errors="coerce")
    keep.append(ch[ids.isin(wanted)])
prod_raw = pd.concat(keep, ignore_index=True)

products_df, p_rep = clean_products(prod_raw)
valid_ids = set(int(x) for x in products_df["product_id"].dropna())
comments_df, c_rep = clean_comments(com_raw, valid_ids)
print("products:", len(products_df), "| comments:", len(comments_df))
print("product report:", p_rep); print("comment report:", c_rep)

products: 17043 | comments: 20000
product report: {'input_rows': 25064, 'dropped_exact_duplicates': 6321, 'dropped_duplicate_ids': 1700, 'zero_or_missing_price': 10, 'output_rows': 17043}
comment report: {'input_rows': 20000, 'dropped_exact_duplicates': 0, 'dropped_duplicate_ids': 0, 'rows_without_text': 0, 'invalid_recommendation_labels': 2892, 'rows_without_product_match': 0, 'output_rows': 20000}


### 1.5 EDA (Plotly)

In [6]:
import json
import logging

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


log = logging.getLogger("digikala.eda")
_TEMPLATE = "plotly_white"


# ---- headline numbers ---------------------------------------------------
def summary_stats(products: pd.DataFrame, comments: pd.DataFrame) -> dict:
    return {
        "n_products": int(len(products)),
        "n_comments": int(len(comments)),
        "n_brands": int(products["brand_norm"].nunique()),
        "n_categories": int(products["category1_norm"].nunique()),
        "pct_products_priced": round(100 * products["price_available"].mean(), 1),
        "pct_comments_with_text": round(100 * comments["has_text"].mean(), 1),
        "pct_comments_labeled": round(100 * comments["recommendation_valid"].mean(), 1),
        "median_price_toman": (None if products["price_clean"].dropna().empty
                               else int(products.loc[products["price_available"], "price_clean"].median() // 10)),
        "avg_comments_per_product": round(
            len(comments) / max(1, comments["product_id"].nunique()), 1),
    }


# ---- figures ------------------------------------------------------------
def fig_recommendation_balance(comments: pd.DataFrame) -> go.Figure:
    vc = comments.loc[comments["recommendation_valid"], "recommendation_status"].value_counts()
    fig = px.bar(x=vc.index, y=vc.values, template=_TEMPLATE,
                 labels={"x": "recommendation_status", "y": "count"},
                 title="Phase 3 target — recommendation_status balance", text=vc.values)
    fig.update_traces(marker_color=["#2ca02c", "#d62728", "#7f7f7f"][:len(vc)])
    return fig


def fig_top_categories(products: pd.DataFrame, n: int = 15) -> go.Figure:
    vc = products["category1_norm"].value_counts().head(n)[::-1]
    return px.bar(x=vc.values, y=vc.index, orientation="h", template=_TEMPLATE,
                  labels={"x": "products", "y": "category"},
                  title=f"Top {n} level-1 categories")


def fig_top_brands(products: pd.DataFrame, n: int = 15) -> go.Figure:
    vc = products.loc[products["brand_norm"] != "نامشخص", "brand_norm"].value_counts().head(n)[::-1]
    return px.bar(x=vc.values, y=vc.index, orientation="h", template=_TEMPLATE,
                  labels={"x": "products", "y": "brand"}, title=f"Top {n} brands")


def fig_price_distribution(products: pd.DataFrame) -> go.Figure:
    priced = products.loc[products["price_available"], "price_clean"] / 10  # Rials -> Toman
    priced = priced[priced > 0]
    fig = px.histogram(np.log10(priced), nbins=60, template=_TEMPLATE,
                       title="Price distribution (log10 Toman)",
                       labels={"value": "log10(price, Toman)"})
    fig.update_layout(showlegend=False)
    return fig


def fig_rating_distribution(products: pd.DataFrame) -> go.Figure:
    rate = products["product_rate_clean"].dropna()
    return px.histogram(rate, nbins=50, template=_TEMPLATE,
                        title="Product rating distribution (0–100)",
                        labels={"value": "product_rate_clean"})


def fig_comments_per_product(comments: pd.DataFrame) -> go.Figure:
    counts = comments.groupby("product_id").size()
    counts = counts[counts > 0]
    fig = px.histogram(np.log10(counts), nbins=50, template=_TEMPLATE,
                       title="Reviews per product (log10)",
                       labels={"value": "log10(reviews per product)"})
    fig.update_layout(showlegend=False)
    return fig


def fig_text_length(comments: pd.DataFrame) -> go.Figure:
    lens = comments.loc[comments["has_text"], "comment_text_norm"].str.split().map(len)
    lens = lens[lens <= lens.quantile(0.99)]        # trim the long tail for readability
    return px.histogram(lens, nbins=60, template=_TEMPLATE,
                        title="Review length (words, 99th pct clipped)",
                        labels={"value": "words per review"})


def fig_missingness(products: pd.DataFrame, comments: pd.DataFrame) -> go.Figure:
    rows = []
    for name, df in [("products", products), ("comments", comments)]:
        for col in df.columns:
            rows.append({"table": name, "column": col,
                         "missing_%": round(100 * df[col].isna().mean(), 1)})
    m = pd.DataFrame(rows)
    m = m[m["missing_%"] > 0].sort_values("missing_%")
    return px.bar(m, x="missing_%", y="column", color="table", orientation="h",
                  template=_TEMPLATE, title="Missing values by column")


ALL_FIGURES = {
    "recommendation_balance": fig_recommendation_balance,
    "top_categories": fig_top_categories,
    "top_brands": fig_top_brands,
    "price_distribution": fig_price_distribution,
    "rating_distribution": fig_rating_distribution,
    "comments_per_product": fig_comments_per_product,
    "text_length": fig_text_length,
    "missingness": fig_missingness,
}


def run() -> dict:
    """Load the cleaned tables, build every figure, and write them to artifacts."""
    products = pd.read_parquet(config.PRODUCTS_CLEAN)
    comments = pd.read_parquet(config.COMMENTS_CLEAN)
    stats = summary_stats(products, comments)

    for name, fn in ALL_FIGURES.items():
        arg = (products,) if fn in (fig_top_categories, fig_top_brands,
                                    fig_price_distribution, fig_rating_distribution) else \
              (comments,) if fn in (fig_recommendation_balance, fig_comments_per_product,
                                    fig_text_length) else (products, comments)
        try:
            fig = fn(*arg)
            fig.write_html(config.FIGURES_DIR / f"{name}.html", include_plotlyjs="cdn")
        except Exception as e:
            log.warning("figure %s failed: %s", name, e)

    (config.FIGURES_DIR / "eda_summary.json").write_text(
        json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
    log.info("EDA figures + summary written to %s", config.FIGURES_DIR)
    return stats

In [7]:
summary_stats(products_df, comments_df)

{'n_products': 17043,
 'n_comments': 20000,
 'n_brands': 1945,
 'n_categories': 168,
 'pct_products_priced': np.float64(99.9),
 'pct_comments_with_text': np.float64(100.0),
 'pct_comments_labeled': np.float64(85.5),
 'median_price_toman': 79000,
 'avg_comments_per_product': 1.2}

In [8]:
fig_recommendation_balance(comments_df).show()
fig_price_distribution(products_df).show()
fig_top_categories(products_df).show()

## 2 · Phase 2 — Grounded shopping assistant
### 2.1 LLM wrapper (3 modes + budget)

In [9]:
import json
import logging
import time


log = logging.getLogger("digikala.llm")

_RETRYABLE_STATUSES = {429, 500, 502, 503, 504}


class BudgetTracker:
    """Tracks hosted-API attempts/successes/failures, tokens, and $ spend."""

    def __init__(self, cap_usd: float = config.BUDGET_USD, log_path=config.BUDGET_LOG):
        self.cap = cap_usd
        self.log_path = log_path
        self.attempted_calls = 0
        self.successful_calls = 0
        self.failed_calls = 0
        self.in_tokens = 0
        self.out_tokens = 0
        self.spent = 0.0
        self.estimated_list_cost = 0.0

    @property
    def calls(self) -> int:                            # back-compat alias
        return self.successful_calls

    def can_spend(self) -> bool:
        return self.spent < self.cap

    def _write(self, row: dict):
        try:
            with open(self.log_path, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        except OSError:
            pass

    def record_attempt(self, provider: str, model: str):
        self.attempted_calls += 1
        self._write({"t": time.time(), "event": "attempt", "provider": provider, "model": model})

    def record_failure(self, provider: str, model: str, status=None, error: str = ""):
        self.failed_calls += 1
        self._write({"t": time.time(), "event": "failure", "provider": provider, "model": model,
                     "status": status, "error": str(error)[:300]})

    def record_success(self, in_tok: int, out_tok: int, *, provider: str, model: str,
                        price_per_m=(0.0, 0.0), billed: bool = False) -> float:
        self.successful_calls += 1
        self.in_tokens += int(in_tok or 0)
        self.out_tokens += int(out_tok or 0)
        estimated = (int(in_tok or 0) * float(price_per_m[0])
                    + int(out_tok or 0) * float(price_per_m[1])) / 1e6
        actual = estimated if billed else 0.0
        self.estimated_list_cost += estimated
        self.spent += actual
        self._write({"t": time.time(), "event": "success", "provider": provider, "model": model,
                     "in": int(in_tok or 0), "out": int(out_tok or 0),
                     "estimated_list_cost": round(estimated, 6), "tracked_cost": round(actual, 6),
                     "billed": bool(billed)})
        return actual

    def summary(self) -> dict:
        resolved = self.successful_calls + self.failed_calls
        return {"api_attempts": self.attempted_calls, "successful_calls": self.successful_calls,
                "failed_calls": self.failed_calls, "resolved_attempts": resolved,
                "unresolved_attempts": max(0, self.attempted_calls - resolved),
                "input_tokens": self.in_tokens, "output_tokens": self.out_tokens,
                "total_cost_usd": round(self.spent, 6),
                "estimated_list_cost_usd": round(self.estimated_list_cost, 6),
                "budget_usd": self.cap, "remaining_usd": round(self.cap - self.spent, 6)}


class LLM:
    """Backend-agnostic chat model. Resolves its provider from `mode` + config."""

    def __init__(self, mode: str | None = None, budget: BudgetTracker | None = None,
                 temperature: float = config.LLM_TEMPERATURE,
                 max_tokens: int = config.LLM_MAX_NEW_TOKENS):
        self.mode = mode or config.RUN_MODE
        self.budget = budget or BudgetTracker()
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.last_cost_usd = 0.0
        self._cache: dict = {}
        self._hf = None
        self.hosted_disabled_reason = None
        self.preferred_network_path = None
        self.provider = self._resolve_provider()

    def _provider_with_key(self, provider_name: str) -> dict:
        import os
        p = dict(config.PROVIDERS[provider_name])
        p["provider_name"] = provider_name
        p["api_key"] = os.environ.get(p["api_key_env"], "")
        return p

    def _resolve_provider(self) -> dict | None:
        if self.mode == "hosted_auto":
            for name in config.HOSTED_PROVIDER_ORDER:
                p = self._provider_with_key(name)
                if p["api_key"]:
                    p["auto_selected"] = True
                    return p
            return {"provider_name": None, "model": None, "base_url": None, "api_key": "",
                    "auto_selected": False, "price_per_m": (0.0, 0.0), "billed": False}
        if self.mode == "free":
            return self._provider_with_key(config.FREE_PROVIDER)
        if self.mode == "paid":
            return self._provider_with_key("paid")
        return None

    @property
    def backend(self) -> str:
        """Human label of what will actually run (for logging/eval)."""
        if self.mode == "local":
            return config.LOCAL_BACKEND
        if self.mode in ("hosted_auto", "free", "paid") and self.provider:
            return f"{self.mode}:{self.provider.get('model')}"
        return self.mode

    def available(self) -> bool:
        if self.mode == "extractive":
            return False
        if self.mode in ("hosted_auto", "free", "paid"):
            return bool(self.provider and self.provider.get("api_key"))
        return True

    # ---- diagnostics (no key ever printed) -------------------------------
    def diagnose(self) -> dict:
        """Check the hosted provider without exposing the API key: try listing
        models, and if that's inconclusive, run one tiny, fully-accounted Chat
        Completions probe. Used before a real run so a dead key / wrong model /
        network issue is caught with a bounded number of requests."""
        if self.mode not in ("hosted_auto", "free", "paid"):
            return {"mode": self.mode, "hosted": False, "available": self.available()}
        p = self.provider or {}
        result = {"mode": self.mode, "provider": p.get("provider_name"), "model": p.get("model"),
                  "key_present": bool(p.get("api_key")), "models_status": None,
                  "model_listed": None, "chat_probe_attempted": False,
                  "chat_probe_success": False, "network_path": None, "error": None}
        if not p.get("api_key"):
            result["error"] = "No recognized hosted API key found in the environment (.env)."
            return result
        import requests
        for path_name, trust_env in (("environment", True), ("direct_no_env_proxy", False)):
            session = requests.Session()
            session.trust_env = trust_env
            try:
                r = session.get(f"{p['base_url']}/models",
                                headers={"Authorization": f"Bearer {p['api_key']}"},
                                timeout=(config.API_CONNECT_TIMEOUT_S, 30))
                result["models_status"] = int(r.status_code)
                result["network_path"] = path_name
                if r.ok:
                    ids = {x.get("id") for x in r.json().get("data", []) if isinstance(x, dict)}
                    result["model_listed"] = p.get("model") in ids
                    self.preferred_network_path = path_name
                    if result["model_listed"]:
                        return result
                    result["error"] = "Configured model not in the provider's model list."
                    break
                result["error"] = r.text[:300]
                if r.status_code == 401:
                    return result
            except requests.RequestException as e:
                result["error"] = str(e)[:300]
            finally:
                session.close()
        # bounded tiny chat probe (fully accounted for in BudgetTracker)
        result["chat_probe_attempted"] = True
        text = self.generate("You are a diagnostic probe.", "Reply with exactly: OK")
        result["chat_probe_success"] = bool(text)
        if text:
            result["error"] = None
        return result

    # ---- generation -------------------------------------------------------
    def generate(self, system: str, user: str) -> str | None:
        """Return the model's reply, or None to signal the extractive fallback."""
        self.last_cost_usd = 0.0
        if not self.available() or self.hosted_disabled_reason:
            return None
        key = (self.mode, system, user)
        if key in self._cache:
            return self._cache[key]
        try:
            if self.mode == "local" and config.LOCAL_BACKEND == "ollama":
                text = self._ollama(system, user)
            elif self.mode == "local":
                text = self._hf_generate(system, user)
            else:
                text = self._openai_compatible(system, user)
        except Exception as e:                       # never crash the pipeline on an LLM error
            log.warning("LLM error (%s), falling back to extractive: %s", self.backend, str(e)[:200])
            return None
        self._cache[key] = text
        return text

    # -- local transformers --
    def _load_hf(self):
        if self._hf is None:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            log.info("loading %s", config.HF_LLM_MODEL)
            tok = AutoTokenizer.from_pretrained(config.HF_LLM_MODEL)
            model = AutoModelForCausalLM.from_pretrained(
                config.HF_LLM_MODEL, dtype=torch.float16,
                device_map="cuda" if torch.cuda.is_available() else "cpu")
            self._hf = (tok, model)
        return self._hf

    def _hf_generate(self, system: str, user: str) -> str:
        import torch
        tok, model = self._load_hf()
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=self.max_tokens, do_sample=False,
                                 temperature=None, top_p=None, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def _ollama(self, system: str, user: str) -> str:
        import requests
        r = requests.post(f"{config.OLLAMA_BASE_URL}/api/chat", timeout=300,
                          json={"model": config.OLLAMA_MODEL, "stream": False,
                                "options": {"temperature": self.temperature, "num_predict": self.max_tokens},
                                "messages": [{"role": "system", "content": system},
                                             {"role": "user", "content": user}]})
        r.raise_for_status()
        return r.json()["message"]["content"].strip()

    # -- hosted OpenAI-compatible (free / paid / hosted_auto) --
    def _openai_compatible(self, system: str, user: str) -> str:
        import requests
        p = self.provider
        provider_name = p.get("provider_name", "")
        chargeable = bool(p.get("billed", False))
        if chargeable and not self.budget.can_spend():
            raise RuntimeError("budget cap reached; refusing a chargeable API call")

        payload = {"model": p["model"], "temperature": self.temperature,
                  "max_tokens": self.max_tokens,
                  "messages": [{"role": "system", "content": system},
                               {"role": "user", "content": user}]}

        last_error: Exception | None = None
        max_rounds = max(1, int(config.API_MAX_ATTEMPTS))
        for round_idx in range(1, max_rounds + 1):
            if self.preferred_network_path == "direct_no_env_proxy":
                paths = [("direct_no_env_proxy", False)]
            elif self.preferred_network_path == "environment":
                paths = [("environment", True)]
            else:
                paths = [("environment", True), ("direct_no_env_proxy", False)]

            for path_name, trust_env in paths:
                if chargeable and not self.budget.can_spend():
                    raise RuntimeError("budget cap reached during retry loop")
                session = requests.Session()
                session.trust_env = trust_env
                self.budget.record_attempt(provider_name, p["model"])
                try:
                    r = session.post(f"{p['base_url']}/chat/completions",
                                     timeout=(config.API_CONNECT_TIMEOUT_S, config.API_READ_TIMEOUT_S),
                                     headers={"Authorization": f"Bearer {p['api_key']}",
                                              "Content-Type": "application/json"},
                                     json=payload)
                except requests.RequestException as e:
                    self.budget.record_failure(provider_name, p["model"], status=None,
                                               error=f"{path_name}: {str(e)[:250]}")
                    last_error = RuntimeError(f"network error via {path_name}: {str(e)[:250]}")
                    session.close()
                    continue                          # try the next network path immediately
                finally:
                    session.close()

                if r.ok:
                    data = r.json()
                    usage = data.get("usage", {})
                    self.preferred_network_path = path_name
                    self.last_cost_usd = self.budget.record_success(
                        usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0),
                        provider=provider_name, model=p["model"],
                        price_per_m=p.get("price_per_m", config.PAID_PRICE_PER_M),
                        billed=chargeable)
                    return str(data["choices"][0]["message"].get("content", "") or "").strip()

                self.budget.record_failure(provider_name, p["model"], status=int(r.status_code),
                                           error=f"{path_name}: {r.text[:250]}")
                last_error = RuntimeError(f"HTTP {r.status_code} via {path_name}: {r.text[:250]}")
                if r.status_code in (400, 401, 404, 422):
                    raise last_error               # not retryable: bad request/auth/model
                if r.status_code == 403 and path_name == "environment":
                    continue                        # 403 can be network/IP-dependent; try direct
                if r.status_code == 403 and path_name == "direct_no_env_proxy":
                    self.hosted_disabled_reason = "HTTP 403 on the direct path; using extractive fallback."
                    raise last_error
                if r.status_code not in _RETRYABLE_STATUSES:
                    raise last_error
                break                                # retryable: end this round, retry next round

            if round_idx < max_rounds:
                time.sleep(float(config.API_RETRY_BASE_S) * round_idx)

        raise last_error or RuntimeError("hosted API call failed")


def judge_llm(budget: BudgetTracker | None = None) -> "LLM":
    """The model Phase 4 uses as LLM-as-judge, chosen by config.JUDGE_MODE."""
    mode = "extractive" if config.JUDGE_MODE == "none" else config.JUDGE_MODE
    return LLM(mode=mode, budget=budget)

### 2.2 Retrieval — dense + BM25 fused with RRF

In [10]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd


log = logging.getLogger("digikala.retrieval")
_model = None


# ---- embedding model (lazy, GPU with CPU fallback) ----------------------
def get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        device = config.EMBEDDING_DEVICE
        try:
            import torch
            if device == "cuda" and not torch.cuda.is_available():
                device = "cpu"
        except Exception:
            device = "cpu"
        log.info("loading %s on %s", config.EMBEDDING_MODEL, device)
        _model = SentenceTransformer(config.EMBEDDING_MODEL, device=device)
    return _model


def embed(texts, batch_size: int = 256) -> np.ndarray:
    texts = [t or "" for t in texts]
    if not texts:
        return np.zeros((0, 384), dtype="float32")
    return get_model().encode(texts, batch_size=batch_size, convert_to_numpy=True,
                              normalize_embeddings=True,
                              show_progress_bar=len(texts) > 5000).astype("float32")


# ---- BM25 (Okapi) — the sparse half of hybrid retrieval -----------------
class BM25Okapi:
    """Okapi BM25 backed by a scipy-sparse term-document matrix.

    A CountVectorizer builds the doc×term counts once (C-optimized), so this scales
    to ~1M products: scoring a query touches only the columns of its terms, and the
    whole index (matrix + idf + doc lengths + vocab) saves/loads from disk in
    seconds instead of rebuilding a Python inverted index every process start.
    """

    def __init__(self, counts, doc_len, idf, vocab, k1: float = 1.5, b: float = 0.75):
        from scipy.sparse import csc_matrix
        self.k1, self.b = k1, b
        self.counts = counts.tocsc() if not isinstance(counts, csc_matrix) else counts
        self.doc_len = doc_len.astype(np.float32)
        self.n = self.counts.shape[0]
        self.avgdl = float(self.doc_len.mean()) if self.n else 1.0
        self.idf = idf.astype(np.float32)
        self.vocab = vocab                            # term -> column index

    @classmethod
    def from_texts(cls, texts, k1: float = 1.5, b: float = 0.75):
        from sklearn.feature_extraction.text import CountVectorizer
        # corpus texts are already normalized (*_norm columns) -> fast regex tokenizer
        vec = CountVectorizer(tokenizer=pt.tokenize_norm, token_pattern=None,
                              lowercase=False, preprocessor=None)
        counts = vec.fit_transform(texts)             # docs x terms, CSR
        doc_len = np.asarray(counts.sum(axis=1)).ravel()
        df = np.asarray((counts > 0).sum(axis=0)).ravel()
        n = counts.shape[0]
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5))
        return cls(counts, doc_len, idf, vec.vocabulary_, k1, b)

    def get_scores(self, query) -> np.ndarray:
        scores = np.zeros(self.n, dtype=np.float32)
        avgdl = max(self.avgdl, 1e-6)
        denom_len = self.k1 * (1 - self.b + self.b * self.doc_len / avgdl)
        for term in set(pt.tokenize(query)):
            col = self.vocab.get(term)
            if col is None:
                continue
            c = self.counts.getcol(col)               # sparse column of term freqs
            rows = c.indices
            freq = c.data.astype(np.float32)
            scores[rows] += self.idf[col] * freq * (self.k1 + 1) / (freq + denom_len[rows])
        return scores

    def save(self, path):
        from scipy.sparse import save_npz
        path = Path(path)
        save_npz(path / "bm25_counts.npz", self.counts)
        np.save(path / "bm25_doc_len.npy", self.doc_len)
        np.save(path / "bm25_idf.npy", self.idf)
        import json
        (path / "bm25_vocab.json").write_text(json.dumps(self.vocab), encoding="utf-8")

    @classmethod
    def load(cls, path, k1: float = 1.5, b: float = 0.75):
        from scipy.sparse import load_npz
        import json
        path = Path(path)
        counts = load_npz(path / "bm25_counts.npz")
        doc_len = np.load(path / "bm25_doc_len.npy")
        idf = np.load(path / "bm25_idf.npy")
        vocab = json.loads((path / "bm25_vocab.json").read_text(encoding="utf-8"))
        return cls(counts, doc_len, idf, vocab, k1, b)


def rrf_fuse(rank_lists, k: int = 60, weights=None) -> dict:
    """Reciprocal-rank fusion over short candidate lists. Weights are optional
    and let sparse lexical evidence count slightly more for exact product
    attributes while keeping dense semantic retrieval in the mix."""
    if weights is None:
        weights = [1.0] * len(rank_lists)
    scores: dict = {}
    for ranks, weight in zip(rank_lists, weights):
        for rank, idx in enumerate(ranks):
            scores[idx] = scores.get(idx, 0.0) + float(weight) / (k + rank + 1)
    return scores


def _minmax(v):
    lo, hi = v.min(), v.max()
    return np.zeros_like(v) if hi - lo < 1e-9 else (v - lo) / (hi - lo)


def product_filter_mask(products: pd.DataFrame, filters: dict) -> np.ndarray:
    mask = np.ones(len(products), dtype=bool)
    f = filters or {}
    if f.get("category"):
        cat = str(f["category"])
        col_mask = np.zeros(len(products), dtype=bool)
        for col in ("category1_norm", "category2_norm", "sub_category_norm"):
            if col in products:
                col_mask |= products[col].fillna("").map(lambda v: cat in str(v)).to_numpy()
        mask &= col_mask
    if f.get("brand"):
        mask &= products["brand_norm"].fillna("").map(lambda v: str(f["brand"]) in str(v)).to_numpy()
    price = products["price_clean"].to_numpy(dtype=float)
    if f.get("price_min") is not None:
        mask &= np.where(np.isnan(price), False, price >= f["price_min"])
    if f.get("price_max") is not None:
        mask &= np.where(np.isnan(price), False, price <= f["price_max"])
    if f.get("exclude_fake", True) and "is_fake" in products:
        mask &= ~products["is_fake"].fillna(False).astype(bool).to_numpy()
    return mask


# ---- the retriever ------------------------------------------------------
class ProductIndex:
    """Full-catalogue product retrieval: dense vectors + BM25, fused with RRF."""

    def __init__(self, products: pd.DataFrame, vectors: np.ndarray, bm25: BM25Okapi, rrf_k: int = 60):
        self.products = products.reset_index(drop=True)
        # keep as-is (a memmap stays on disk) unless the dtype needs converting
        self.vectors = vectors if vectors.dtype == np.float32 else vectors.astype("float32")
        self.bm25 = bm25
        self.rrf_k = rrf_k

    @classmethod
    def build(cls, products: pd.DataFrame) -> "ProductIndex":
        texts = products["product_text_norm"].fillna("").tolist()
        log.info("embedding %d products", len(texts))
        vectors = embed(texts)
        bm25 = BM25Okapi.from_texts(texts)
        return cls(products, vectors, bm25)

    def save(self, path=config.PRODUCT_INDEX_DIR):
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        np.save(path / "vectors.npy", self.vectors)
        self.products.to_parquet(path / "products.parquet", index=False)
        self.bm25.save(path)                          # persist the sparse BM25 so load is fast
        log.info("saved product index (%d) to %s", len(self.products), path)

    @classmethod
    def load(cls, path=config.PRODUCT_INDEX_DIR) -> "ProductIndex":
        path = Path(path)
        products = pd.read_parquet(path / "products.parquet")
        vectors = np.load(path / "vectors.npy", mmap_mode="r")
        bm25 = BM25Okapi.load(path)
        return cls(products, vectors, bm25)

    def search(self, query: str, filters: dict | None = None, k: int = config.TOP_K,
               method: str = "hybrid") -> list[dict]:
        """method: 'hybrid' (dense+BM25 RRF, default) | 'dense' | 'bm25' -- the
        latter two exist for the retrieval ablation (quantifying hybrid's lift)."""
        mask = product_filter_mask(self.products, filters)
        valid = np.flatnonzero(mask)
        if valid.size == 0:
            return []
        if method == "dense":
            qv = embed([query])[0]
            dense = self.vectors @ qv
            order = valid[np.argsort(-dense[valid], kind="stable")][:k]
            top = [(idx, float(dense[idx])) for idx in order]
        elif method == "bm25":
            sparse = self.bm25.get_scores(query)
            order = valid[np.argsort(-sparse[valid], kind="stable")][:k]
            top = [(idx, float(sparse[idx])) for idx in order]
        else:
            qv = embed([query])[0]
            dense = self.vectors @ qv
            sparse = self.bm25.get_scores(query)
            # cap the candidate pool so RRF stays cheap at ~1M-product scale
            pool = min(int(valid.size), max(int(config.RRF_CANDIDATE_POOL), int(k) * 20))
            d_order = valid[np.argsort(-dense[valid], kind="stable")[:pool]]
            sparse_valid = valid[sparse[valid] > 0]     # drop zero-score lexical noise
            if sparse_valid.size:
                s_order = sparse_valid[np.argsort(-sparse[sparse_valid], kind="stable")[:pool]]
                fused = rrf_fuse([d_order.tolist(), s_order.tolist()], k=self.rrf_k,
                                 weights=[config.PRODUCT_RRF_DENSE_WEIGHT, config.PRODUCT_RRF_SPARSE_WEIGHT])
            else:
                fused = rrf_fuse([d_order.tolist()], k=self.rrf_k)
            top = sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:k]
        out = []
        for idx, score in top:
            r = self.products.iloc[idx]
            out.append({
                "rank": len(out) + 1, "product_id": int(r["product_id"]),
                "title": r["title_fa"], "brand": r["brand_norm"],
                "price": float(r["price_clean"]) if pd.notna(r["price_clean"]) else None,
                "rate": float(r["product_rate_clean"]) if pd.notna(r["product_rate_clean"]) else None,
                "rate_count": int(r["rate_count"]) if pd.notna(r["rate_count"]) else 0,
                "comment_count": int(r.get("comment_count", 0)),
                "category": r.get("category1_norm", ""), "score": round(float(score), 5),
            })
        return out


class ReviewRetriever:
    """Per-product hybrid review search with query-intent-aware reranking.

    Beyond plain RRF, reviews are reranked by whether the QUERY is asking about
    problems ("مشکل چیه؟") vs. satisfaction ("خوبه؟") — a negative-intent query
    should surface reviews that actually read negative (low rating / not
    recommended / has a disadvantages field), not just whatever ranked highest
    on raw text similarity. This directly improves QA precision: without it, a
    "what's wrong with this product" query could return the top-BM25-scoring
    review even if it happens to be glowingly positive.
    """

    _NEGATIVE_CUES = ("مشکل", "ایراد", "عیب", "بد", "ضعف", "منفی", "ناراضی",
                      "خراب", "معیوب", "نمی ارزد", "نمی‌ارزد", "عدم توصیه",
                      "نقاط ضعف", "معایب")
    _POSITIVE_CUES = ("خوب", "مثبت", "مزیت", "مزایا", "رضایت", "راضی",
                      "ارزش خرید", "پیشنهاد", "نقاط قوت")

    def __init__(self, comments_by_product, rrf_k: int = 60):
        self.by_product = comments_by_product
        self.rrf_k = rrf_k

    @staticmethod
    def _rating_norm(rate: np.ndarray) -> np.ndarray:
        rate = np.asarray(rate, dtype=float)
        if rate.size == 0:
            return rate
        finite = np.isfinite(rate)
        if not finite.any():
            return np.full(len(rate), 0.5, dtype=float)
        vals = rate.copy()
        vals[~finite] = float(np.nanmedian(vals[finite]))
        lo, hi = float(np.min(vals)), float(np.max(vals))
        if hi - lo < 1e-9:
            return np.full(len(vals), 0.5, dtype=float)
        return (vals - lo) / (hi - lo)

    def _polarity(self, query: str) -> str:
        q = pt.normalize(query)
        if any(x in q for x in self._NEGATIVE_CUES):
            return "negative"
        if any(x in q for x in self._POSITIVE_CUES):
            return "positive"
        return "neutral"

    def retrieve(self, query: str, product_id: int, k: int = config.TOP_K, rerank: bool = True) -> list[dict]:
        rev = self.by_product.get(int(product_id))
        if rev is None or rev.empty:
            return []
        rev = rev.reset_index(drop=True)
        texts = rev["comment_text_norm"].fillna("").tolist()
        qv = embed([query])[0]
        dense = embed(texts) @ qv
        sparse = BM25Okapi.from_texts(texts).get_scores(query)

        pool = min(len(rev), max(int(config.REVIEW_CANDIDATE_POOL), int(k) * 10))
        order_d = np.argsort(-dense, kind="stable")[:pool].tolist()
        positive_sparse = np.flatnonzero(sparse > 0)
        if positive_sparse.size:
            order_s = positive_sparse[np.argsort(-sparse[positive_sparse], kind="stable")[:pool]].tolist()
            fused = rrf_fuse([order_d, order_s], k=self.rrf_k, weights=[1.0, 1.05])
        else:
            fused = rrf_fuse([order_d], k=self.rrf_k)

        if rerank and fused:
            idxs = list(fused)
            base = _minmax(np.array([fused[i] for i in idxs], dtype=float))
            likes = rev["likes"].fillna(0).to_numpy(dtype=float)
            buyer = rev["is_buyer"].fillna(False).astype(float).to_numpy()
            lk = _minmax(np.log1p(likes[idxs]))
            rate_all = pd.to_numeric(rev["rate_clean"], errors="coerce").to_numpy(dtype=float)
            rate_norm = self._rating_norm(rate_all)
            status = rev["recommendation_status"].fillna("").astype(str).to_numpy()
            has_disadv = (rev["disadvantages_norm"].fillna("").astype(str).str.len().gt(0).to_numpy()
                         if "disadvantages_norm" in rev else np.zeros(len(rev), dtype=bool))
            has_adv = (rev["advantages_norm"].fillna("").astype(str).str.len().gt(0).to_numpy()
                      if "advantages_norm" in rev else np.zeros(len(rev), dtype=bool))

            polarity = self._polarity(query)
            intent_score = np.full(len(idxs), 0.5, dtype=float)
            if polarity == "negative":
                for j, idx in enumerate(idxs):
                    status_score = 1.0 if status[idx] == "not_recommended" else (0.5 if status[idx] == "no_idea" else 0.0)
                    intent_score[j] = 0.55 * status_score + 0.30 * (1.0 - rate_norm[idx]) + 0.15 * float(has_disadv[idx])
            elif polarity == "positive":
                for j, idx in enumerate(idxs):
                    status_score = 1.0 if status[idx] == "recommended" else (0.5 if status[idx] == "no_idea" else 0.0)
                    intent_score[j] = 0.60 * status_score + 0.25 * rate_norm[idx] + 0.15 * float(has_adv[idx])

            final_scores = {}
            for j, idx in enumerate(idxs):
                if polarity == "neutral":
                    score = 0.84 * base[j] + 0.09 * lk[j] + 0.07 * buyer[idx]
                else:
                    iw = float(config.REVIEW_NEGATIVE_INTENT_WEIGHT if polarity == "negative"
                              else config.REVIEW_POSITIVE_INTENT_WEIGHT)
                    score = (0.93 - iw) * base[j] + 0.07 * lk[j] + 0.03 * buyer[idx] + iw * intent_score[j]
                final_scores[idx] = float(score)
            fused = final_scores

        top = sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:k]
        out = []
        for idx, score in top:
            r = rev.iloc[idx]
            out.append({
                "rank": len(out) + 1, "comment_id": int(r["comment_id"]),
                "product_id": int(r["product_id"]), "text": r["comment_text_norm"],
                "rate": float(r["rate_clean"]) if pd.notna(r["rate_clean"]) else None,
                "recommendation_status": str(r["recommendation_status"]),
                "likes": int(r["likes"]) if pd.notna(r["likes"]) else 0,
                "is_buyer": bool(r["is_buyer"]) if pd.notna(r["is_buyer"]) else False,
                "has_advantage": bool(str(r.get("advantages_norm", "") or "").strip()),
                "has_disadvantage": bool(str(r.get("disadvantages_norm", "") or "").strip()),
                "score": round(float(score), 5),
            })
        return out


# ---- build helpers ------------------------------------------------------
def _prepare_products(products: pd.DataFrame, comments: pd.DataFrame) -> pd.DataFrame:
    products = products.copy()
    counts = comments.groupby("product_id").size()
    products["comment_count"] = products["product_id"].map(counts).fillna(0).astype(int)
    return products


def build_product_index(sample_comments: int | None = None) -> ProductIndex:
    """Build + persist the product index from the cleaned tables (run.py index)."""
    products = pd.read_parquet(config.PRODUCTS_CLEAN)
    comments = pd.read_parquet(config.COMMENTS_CLEAN, columns=["product_id"])
    products = _prepare_products(products, comments)
    idx = ProductIndex.build(products)
    idx.save()
    return idx


class GroupedComments:
    """Per-product review access backed by ONE index-sorted DataFrame.

    Behaves like a dict of {product_id: reviews-DataFrame} (`.get`, `in`, iteration,
    `len`) but never materializes a separate frame per product, so it stays memory-
    lean on the full corpus (millions of reviews) instead of holding N sub-frames.
    """

    def __init__(self, df: pd.DataFrame):
        df = df.dropna(subset=["product_id"]).copy()
        df["product_id"] = df["product_id"].astype(int)
        self.df = df.set_index("product_id", drop=False).sort_index()
        self._ids = set(self.df.index.unique())

    def get(self, pid, default=None):
        pid = int(pid)
        if pid not in self._ids:
            return default
        # reset to a clean RangeIndex — callers do .loc/.assign that break on the
        # duplicated product_id index
        return self.df.loc[[pid]].reset_index(drop=True)

    def __contains__(self, pid):
        return int(pid) in self._ids

    def __iter__(self):
        return iter(self._ids)

    def __len__(self):
        return len(self._ids)


_REVIEW_COLS = ["comment_id", "product_id", "comment_text_norm", "body_norm",
                "advantages_norm", "disadvantages_norm", "recommendation_status",
                "rate_clean", "likes", "is_buyer", "has_text"]


def load_comments_by_product(only_with_text: bool = True) -> GroupedComments:
    """Load cleaned comments (needed columns only) for on-demand review retrieval."""
    import pyarrow.parquet as pq
    have = set(pq.ParquetFile(config.COMMENTS_CLEAN).schema.names)
    cols = [c for c in _REVIEW_COLS if c in have]
    comments = pd.read_parquet(config.COMMENTS_CLEAN, columns=cols)
    if only_with_text and "has_text" in comments:
        comments = comments[comments["has_text"].astype(bool)]
    return GroupedComments(comments)

retrieval = types.SimpleNamespace(get_model=get_model, embed=embed, BM25Okapi=BM25Okapi,
    rrf_fuse=rrf_fuse, product_filter_mask=product_filter_mask, ProductIndex=ProductIndex,
    ReviewRetriever=ReviewRetriever, GroupedComments=GroupedComments,
    _prepare_products=_prepare_products, load_comments_by_product=load_comments_by_product)

### 2.3 Intent router + entity resolution

In [11]:
import re
from dataclasses import dataclass, field

import pandas as pd


_GENERIC = {"متفرقه", "نامشخص", "سایر"}
_CMP = ("مقایسه", "تفاوت", "فرق", "کدوم بهتر", "کدام بهتر", "بهتره", "بهتر است", "vs", "در برابر")
_QA = ("آیا", "چطور", "چگونه", "چرا", "چند", "چقدر", "مشکل", "سایز", "اندازه", "جنس",
       "کیفیت", "مناسب", "خوب", "بد", "رضایت", "ارزش خرید", "باتری", "دوام", "نظر")
# NOTE: deliberately no bare "دسته" ("category") here -- `scope` already detects
# a category/brand mention, so if "دسته" alone counted as an analytical cue,
# every discovery request that names a category ("در دستهٔ X چند محصول خوب
# هست؟") would incorrectly route to managerial. These cues represent genuine
# analytical/complaint intent, not just "a category was named".
_MNG = ("شکایت", "شکایات", "عملکرد", "تحلیل", "برند", "مشکلات", "پرفروش",
        "کمترین", "بیشترین", "محبوب", "نرخ", "نارضایتی", "بازار", "مدیر")


@dataclass
class Catalog:
    products: pd.DataFrame
    comments_by_product: dict
    category_values: dict = field(default_factory=dict)
    brand_values: set = field(default_factory=set)
    _by_id: pd.DataFrame = None                       # products indexed by product_id
    reviewed_title_tokens: dict = field(default_factory=dict)  # pid -> token set (reviewed only)

    @classmethod
    def build(cls, products: pd.DataFrame, comments_by_product: dict) -> "Catalog":
        products = products.reset_index(drop=True)
        by_id = products.dropna(subset=["product_id"]).copy()
        by_id["product_id"] = by_id["product_id"].astype(int)
        by_id = by_id.set_index("product_id", drop=False)
        cats = {c: {v for v in products[c].dropna().unique() if str(v).strip() and str(v) not in _GENERIC}
                for c in ("category1_norm", "category2_norm", "sub_category_norm") if c in products}
        brands = {v for v in products["brand_norm"].dropna().unique() if str(v).strip() and str(v) not in _GENERIC}
        # only reviewed products are candidates for fuzzy name->id resolution (QA needs reviews)
        reviewed = {}
        for pid in comments_by_product:
            if pid in by_id.index:
                reviewed[int(pid)] = set(pt.tokenize_norm(by_id.at[int(pid), "title_norm"]))
        return cls(products, comments_by_product, cats, brands, by_id, reviewed)

    @property
    def product_lookup(self):                          # kept for compatibility with callers
        return self._by_id

    def product(self, pid):
        pid = int(pid)
        if self._by_id is not None and pid in self._by_id.index:
            return self._by_id.loc[pid].to_dict()
        return None


def match_known_value(query, values, min_ratio: float = 0.6):
    # `values` come from already-normalized *_norm columns, so avoid re-running the
    # (heavy) hazm normalizer per value — normalize the query once, split the rest.
    t = pt.normalize(query)
    t_tokens = set(pt.tokenize_norm(t))
    best, best_score = None, 0.0
    for v in values:
        vn = str(v).strip()
        if not vn or len(vn) < 3 or vn in _GENERIC:
            continue
        v_tokens = set(pt.tokenize_norm(vn))
        if not v_tokens:
            continue
        if vn in t:
            score = len(vn) + 100.0
        else:
            ratio = len(t_tokens & v_tokens) / max(1, len(v_tokens))
            if ratio < min_ratio:
                continue
            score = ratio * len(v_tokens)
        if score > best_score:
            best, best_score = vn, score
    return best


def extract_product_ids(catalog: Catalog, query) -> list[int]:
    t = pt.normalize(query)
    found: list[int] = []
    for m in re.finditer(r"\b(\d{6,9})\b", t):
        pid = int(m.group(1))
        if pid in catalog._by_id.index and pid not in found:
            found.append(pid)
    return found


def resolve_product_id(catalog: Catalog, text):
    """Fuzzy map a description to a product id, searching only reviewed products
    (a Q&A only makes sense for a product that actually has reviews)."""
    t = pt.normalize(text)
    m = re.search(r"\b(\d{6,9})\b", t)
    if m and int(m.group(1)) in catalog._by_id.index:
        return int(m.group(1))
    q_tokens = set(pt.tokenize(t))
    if not q_tokens:
        return None
    best_pid, best = None, 0.0
    for pid, title_tokens in catalog.reviewed_title_tokens.items():
        if not title_tokens:
            continue
        score = len(q_tokens & title_tokens) / max(1, len(title_tokens))
        if score > best:
            best_pid, best = pid, score
    return best_pid if best >= 0.5 else None


def resolve_scope(catalog: Catalog, text) -> dict:
    best, best_score = None, 0.0
    for kind, values in catalog.category_values.items():
        v = match_known_value(text, values)
        if v and len(v) > best_score:
            best, best_score = (kind, v), len(v)
    v = match_known_value(text, catalog.brand_values)
    if v and len(v) > best_score:
        best, best_score = ("brand_norm", v), len(v)
    return {"kind": best[0], "value": best[1]} if best else {}


def extract_filters(catalog: Catalog, query) -> dict:
    filters: dict = {}
    best_cat, best_len = None, 0
    for values in catalog.category_values.values():
        v = match_known_value(query, values)
        if v and len(v) > best_len:
            best_cat, best_len = v, len(v)
    if best_cat:
        filters["category"] = best_cat
    b = match_known_value(query, catalog.brand_values)
    if b:
        filters["brand"] = b
    filters.update(pt.extract_price_constraint(query, config.TOMAN_TO_RIAL))
    return filters


@dataclass
class Route:
    intent: str
    product_id: int | None = None
    product_ids: list = field(default_factory=list)
    scope: dict = field(default_factory=dict)
    filters: dict = field(default_factory=dict)
    needs_clarification: bool = False


class IntentRouter:
    def __init__(self, catalog: Catalog):
        self.c = catalog

    def route(self, query) -> Route:
        t = pt.normalize(query)
        ids = extract_product_ids(self.c, query)
        scope = resolve_scope(self.c, query)
        if len(ids) >= 2 or (any(w in t for w in _CMP) and len(ids) >= 1):
            if len(ids) >= 2:
                return Route("comparison", product_ids=ids)
            return Route("comparison", product_ids=ids, needs_clarification=True)
        has_qa_cue = any(w in t for w in _QA)
        # An explicit numeric id is unambiguous -> product_qa regardless of scope.
        if ids and has_qa_cue:
            return Route("product_qa", product_id=ids[0])
        # A confident managerial-scope match takes priority over the FUZZY (no
        # explicit id) name->product resolution below: fuzzy token-overlap
        # matching can accidentally hit some unrelated reviewed product's title
        # just because a generic QA cue word (like "چند") appears in a query
        # that is actually asking about a whole category, e.g. "در دستهٔ اسباب
        # بازی چند محصول ... معرفی کن" -- caught by the response-eval routing
        # regression check.
        if scope and any(w in t for w in _MNG):
            return Route("managerial", scope=scope)
        if has_qa_cue:
            pid = resolve_product_id(self.c, query)
            if pid is not None:
                return Route("product_qa", product_id=pid)
        return Route("discovery", filters=extract_filters(self.c, query))

### 2.4 Grounded prompts + evidence formatters

In [12]:
SYSTEM_CORE = (
    "تو یک دستیار خرید دیجی‌کالا هستی که فقط بر پایهٔ داده‌های واقعی پاسخ می‌دهی.\n"
    "۱) فقط از «مدارک» استفاده کن؛ دانش قبلی یا عددسازی ممنوع.\n"
    "۲) هر ادعا باید با [محصول شناسه] یا [بازبینی شناسه] ارجاع داده شود.\n"
    "۳) اگر مدارک کافی نیست، صریحاً بنویس «اطلاعات کافی موجود نیست».\n"
    "۴) پاسخ فارسی، کوتاه و ساختاریافته باشد.\n"
    "۵) عددها را عیناً از مدارک کپی کن.\n"
)
DISCOVERY_SYSTEM = SYSTEM_CORE + "\nوظیفه: پیشنهاد و رتبه‌بندی محصول. حداکثر {k} پیشنهاد، هرکدام با [محصول ...] و دلیل کوتاه."
QA_SYSTEM = SYSTEM_CORE + "\nوظیفه: پاسخ به پرسش دربارهٔ یک محصول، فقط از بازبینی‌ها. هر ادعا با [بازبینی ...]. حداکثر {max_lines} خط."
COMPARISON_SYSTEM = SYSTEM_CORE + "\nوظیفه: مقایسهٔ محصولات از جدول واقعیت و بازبینی‌ها؛ نبودِ داده را در «اطلاعات موجود نیست» بگو."
MANAGERIAL_SYSTEM = SYSTEM_CORE + "\nوظیفه: تحلیل مدیریتی (شکایت‌ها، محصولات با نرخ توصیهٔ پایین، رضایت برندها). حداکثر {max_paragraphs} پاراگراف."

# LLM-as-judge prompts (Phase 4)
JUDGE_FAITH_SYS = ("تو یک ارزیاب هستی. «پایبندی به منبع» را بسنج: آیا هر ادعای پاسخ در منابع "
                   "ارجاع‌شده پشتیبانی می‌شود؟ فقط یک عدد ۰ تا ۵ بنویس.")
JUDGE_REL_SYS = ("تو یک ارزیاب هستی. «مفید و مرتبط بودن پاسخ» را بسنج: آیا مستقیم و مرتبط به "
                 "سؤال جواب داده؟ فقط یک عدد ۰ تا ۵ بنویس.")
JUDGE_USER = "سؤال: {query}\n\nپاسخ سیستم:\n{answer}\n\nمنابع:\n{sources}\n\nنمره (۰ تا ۵):"


def evidence_products(rows) -> str:
    out = []
    for r in rows:
        out.append(f"[محصول {r['product_id']}] {r['title']} | برند: {r['brand'] or 'نامشخص'} | "
                   f"قیمت: {format_toman(r['price'])} تومان | امتیاز: {r['rate'] if r['rate'] is not None else 'نامشخص'} | "
                   f"تعداد نظر: {r['comment_count']}")
    return "\n".join(out)


def evidence_reviews(rows) -> str:
    m = {"recommended": "توصیه", "not_recommended": "عدم توصیه", "no_idea": "نظری ندارد"}
    out = []
    for r in rows:
        out.append(f"[بازبینی {r['comment_id']}] (محصول {r['product_id']}) "
                   f"امتیاز {r['rate'] if r['rate'] is not None else '?'} | "
                   f"{m.get(r['recommendation_status'], 'نامشخص')} | پسند {r['likes']}\nمتن: {r['text']}")
    return "\n".join(out)

prompts = types.SimpleNamespace(SYSTEM_CORE=SYSTEM_CORE, DISCOVERY_SYSTEM=DISCOVERY_SYSTEM,
    QA_SYSTEM=QA_SYSTEM, COMPARISON_SYSTEM=COMPARISON_SYSTEM, MANAGERIAL_SYSTEM=MANAGERIAL_SYSTEM,
    JUDGE_FAITH_SYS=JUDGE_FAITH_SYS, JUDGE_REL_SYS=JUDGE_REL_SYS, JUDGE_USER=JUDGE_USER,
    evidence_products=evidence_products, evidence_reviews=evidence_reviews)

### 2.5 The assistant — route → retrieve → answer → verify citations

In [13]:
import re
import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd


# ---- managerial aggregates ---------------------------------------------
_COMPLAINT_TERMS = ["خراب", "عیب", "ایراد", "مشکل", "بوی", "شکست", "شکسته", "پاره",
                    "افتضاح", "پشیمان", "بی کیفیت", "تقلبی", "فیک", "معیوب", "جنس بد",
                    "کیفیت بد", "حساسیت", "جوش", "ترک", "خش", "لک", "چروک", "بو میده"]
_NORM_TERMS = sorted({pt.normalize(t) for t in _COMPLAINT_TERMS if pt.normalize(t)}, key=len, reverse=True)
_GENERIC_VALUES = {"", "نامشخص", "سایر", "متفرقه"}


def _rate_low_threshold(rate: pd.Series) -> float:
    """Digikala rate columns can be on a 0..5-like or 0..100-like scale
    depending on source; pick the threshold that matches the observed range
    instead of assuming one scale (a real bug class if assumed wrong)."""
    finite = pd.to_numeric(rate, errors="coerce").dropna()
    if finite.empty:
        return 2.0
    return 2.0 if float(finite.max()) <= 5.0 else 40.0


def review_stats(catalog: Catalog, product_id, light: bool = False) -> dict:
    """Aggregate one product's reviews. light=True skips the top advantages/
    disadvantages groupby — used by managerial, which ranks products but doesn't
    need per-product pros/cons (much faster over hundreds of products)."""
    rev = catalog.comments_by_product.get(int(product_id))
    if rev is None or rev.empty:
        return {"product_id": int(product_id), "n_reviews": 0, "n_labeled_recommendation": 0,
                "rec_rate": None, "not_rec_rate": None, "no_idea_rate": None, "avg_rate": None,
                "complaint_count": 0, "top_advantages": [], "top_disadvantages": []}
    n = len(rev)
    status = rev["recommendation_status"].fillna("").astype(str)
    valid_status = status.isin(("recommended", "not_recommended", "no_idea"))
    n_labeled = int(valid_status.sum())
    rec = int((status == "recommended").sum())
    not_rec = int((status == "not_recommended").sum())
    no_idea = int((status == "no_idea").sum())
    avg = float(rev["rate_clean"].dropna().mean()) if rev["rate_clean"].notna().any() else None

    def _top(col, k=3):
        if col not in rev:
            return []
        col_s = rev[col].fillna("")
        s = col_s[col_s.str.len() > 0]                # non-empty, no deprecated replace()
        if s.empty:
            return []
        ranked = (s.to_frame().assign(_l=rev.loc[s.index, "likes"].fillna(0))
                  .groupby(col, sort=False)["_l"].sum().sort_values(ascending=False))
        rows = []
        for text in ranked.head(k).index:
            candidates = rev.loc[s[s == text].index].sort_values("likes", ascending=False)
            r = candidates.iloc[0]
            rows.append({"text": str(text), "comment_id": int(r["comment_id"]),
                        "product_id": int(r["product_id"]),
                        "rate": float(r["rate_clean"]) if pd.notna(r["rate_clean"]) else None,
                        "recommendation_status": str(r["recommendation_status"]),
                        "likes": int(r["likes"]) if pd.notna(r["likes"]) else 0,
                        "is_buyer": bool(r["is_buyer"]) if pd.notna(r["is_buyer"]) else False})
        return rows

    low = _rate_low_threshold(rev["rate_clean"])
    neg = (status == "not_recommended") | (pd.to_numeric(rev["rate_clean"], errors="coerce") <= low)
    out = {"product_id": int(product_id), "n_reviews": n, "n_labeled_recommendation": n_labeled,
           "rec_rate": round(rec / n_labeled, 3) if n_labeled else None,
           "not_rec_rate": round(not_rec / n_labeled, 3) if n_labeled else None,
           "no_idea_rate": round(no_idea / n_labeled, 3) if n_labeled else None,
           "avg_rate": round(avg, 1) if avg is not None else None,
           "complaint_count": int(neg.fillna(False).sum()), "top_advantages": [], "top_disadvantages": []}
    if not light:
        out["top_advantages"] = _top("advantages_norm")
        out["top_disadvantages"] = _top("disadvantages_norm")
    return out


def _top_complaint_terms(catalog: Catalog, pids, k: int = 8):
    per = {t: re.compile(r"(?:^|\s)" + re.escape(t) + r"(?:$|\s)") for t in _NORM_TERMS}
    counts: dict = {}
    for pid in pids:
        rev = catalog.comments_by_product.get(int(pid))
        if rev is None:
            continue
        low = _rate_low_threshold(rev["rate_clean"])
        for r in rev.itertuples():
            dis = pt.normalize(getattr(r, "disadvantages_norm", "") or "")
            is_neg = str(r.recommendation_status) == "not_recommended" or (
                pd.notna(r.rate_clean) and float(r.rate_clean) <= low)
            if not is_neg and not dis.strip():
                continue
            text = " " + dis + (" " + pt.normalize(getattr(r, "body_norm", "") or "") if is_neg else "") + " "
            for term, rx in per.items():
                m = len(rx.findall(text))
                if m:
                    counts[term] = counts.get(term, 0) + m
    top = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:k]
    return [{"term": t, "count": c} for t, c in top]


def managerial_summary(catalog: Catalog, scope, min_comments: int = 5) -> dict:
    """Category/brand analytics. Rates are REVIEW-WEIGHTED (not a plain mean of
    per-product rates, which would let a 1-review product outweigh a 500-review
    one), brand satisfaction requires a minimum sample, and the low-recommendation
    threshold falls back to a smaller minimum when the strict one yields nothing
    rather than silently returning an empty list."""
    kind, value = scope.get("kind"), pt.normalize(scope.get("value", ""))
    prods = catalog.products
    if not (kind and kind in prods):
        return {"scope": scope, "n_products": 0}
    scoped = prods[prods[kind].fillna("").str.strip() == value]
    if scoped.empty:
        return {"scope": scope, "n_products": 0}
    reviewed = set(catalog.comments_by_product)
    pids = [int(x) for x in scoped["product_id"].dropna().tolist() if int(x) in reviewed]
    if not pids:
        return {"scope": scope, "n_products": 0}
    rows = []
    for pid in pids:
        s = review_stats(catalog, pid, light=True)
        row = catalog.product(pid) or {}
        s.update({"title": row.get("title_fa", ""), "price": row.get("price_clean"),
                  "brand": row.get("brand_norm", ""), "rate": row.get("product_rate_clean")})
        rows.append(s)
    df = pd.DataFrame(rows)

    total_reviews = int(df["n_reviews"].sum())
    total_complaints = int(df["complaint_count"].sum())
    total_labeled = int(df["n_labeled_recommendation"].sum())
    weighted_rec_rate = (float((df["rec_rate"].fillna(0) * df["n_labeled_recommendation"]).sum() / total_labeled)
                         if total_labeled else None)
    product_mean_rec_rate = float(df["rec_rate"].mean()) if df["rec_rate"].notna().any() else None

    brand_sat = []
    if kind in ("category1_norm", "category2_norm", "sub_category_norm"):
        for brand, g in df.groupby("brand", dropna=True):
            brand = str(brand).strip()
            if not brand or brand in _GENERIC_VALUES:
                continue
            n_reviews_b = int(g["n_reviews"].sum())
            n_labeled_b = int(g["n_labeled_recommendation"].sum())
            if len(g) < 2 or n_reviews_b < 5 or n_labeled_b < 5:
                continue                              # too little data to trust a brand comparison
            rec_rate_b = float((g["rec_rate"].fillna(0) * g["n_labeled_recommendation"]).sum() / n_labeled_b)
            valid_rate = g["avg_rate"].notna()
            avg_rate_b = (float(np.average(g.loc[valid_rate, "avg_rate"],
                                           weights=g.loc[valid_rate, "n_reviews"].clip(lower=1)))
                         if valid_rate.any() else None)
            brand_sat.append({"brand": brand, "n_products": int(len(g)), "n_reviews": n_reviews_b,
                              "n_labeled_recommendation": n_labeled_b,
                              "review_weighted_rec_rate": round(rec_rate_b, 3),
                              "review_weighted_avg_rate": round(avg_rate_b, 2) if avg_rate_b is not None else None})
        brand_sat = sorted(brand_sat, key=lambda x: (x["n_reviews"], x["review_weighted_rec_rate"]),
                           reverse=True)[:8]

    threshold = int(min_comments)
    low = pd.DataFrame()
    for candidate in [threshold, 3, 2]:
        if candidate > threshold:
            continue
        cand_df = df[(df["n_reviews"] >= candidate) & df["rec_rate"].notna()].sort_values(
            ["rec_rate", "n_reviews"], ascending=[True, False])
        if len(cand_df):
            low, threshold = cand_df.head(8), candidate
            break
    low_list = [{"product_id": int(r.product_id), "title": r.title, "rec_rate": r.rec_rate,
                 "n_reviews": int(r.n_reviews), "rate": r.rate, "price": r.price}
                for r in low.itertuples(index=False)]

    return {"scope": scope, "product_ids": pids, "n_products": int(len(df)),
            "n_reviews": total_reviews, "n_labeled_recommendations": total_labeled,
            "total_complaints": total_complaints,
            "complaints_per_100_reviews": round(100 * total_complaints / total_reviews, 1) if total_reviews else None,
            "avg_rate": round(float(df["avg_rate"].mean()), 1) if df["avg_rate"].notna().any() else None,
            "avg_rec_rate_product_mean": round(product_mean_rec_rate, 3) if product_mean_rec_rate is not None else None,
            "review_weighted_rec_rate": round(weighted_rec_rate, 3) if weighted_rec_rate is not None else None,
            # kept for dashboard/back-compat: same as the weighted figure above
            "avg_rec_rate": round(weighted_rec_rate, 3) if weighted_rec_rate is not None else None,
            "low_recommendation_min_reviews": int(threshold) if len(low) else None,
            "brand_satisfaction": brand_sat, "low_recommendation_products": low_list,
            "top_complaint_terms": _top_complaint_terms(catalog, pids)}


# ---- comparison/QA evidence quality guards -------------------------------
# A raw disadvantages/advantages field is often just "ندارد" ("none") -- citing
# that as if it were real evidence, or citing an empty/wrong-polarity string as
# a "weakness", is a real correctness bug the reference implementation caught.
_PROCON_EMPTY_PHRASES = {"", "ندارد", "نداره", "ندارم", "نداشت", "نداشتم", "هیچ", "هیچی",
                         "موردی ندارد", "مورد خاصی ندارد", "نکته منفی ندارد",
                         "نقطه ضعف ندارد", "عیبی ندارد", "ایرادی ندارد", "مشکلی ندارد"}
_POSITIVE_EVIDENCE_HINTS = ("خوب", "عالی", "راضی", "رضایت", "مناسب", "جذاب", "ارزش خرید",
                            "پیشنهاد", "با کیفیت", "باکیفیت", "خوشش اومد", "خوشش آمد")
_STRONG_NEGATIVE_EVIDENCE_HINTS = ("بد", "ضعیف", "مشکل", "ایراد", "عیب", "خراب", "معیوب",
                                   "ناراضی", "نامناسب", "تقلبی", "فیک", "شکسته", "شکست",
                                   "پاره", "افتضاح", "بی کیفیت", "بی‌کیفیت", "کیفیت پایین",
                                   "نمی ارزد", "نمی‌ارزد", "ارزش نداره", "ارزش ندارد")


def _procon_flags(row: dict) -> dict:
    text = pt.normalize(row.get("text", "") or "")
    compact = re.sub(r"[\s،,؛;.!؟?]+", " ", text).strip()
    placeholder = compact in _PROCON_EMPTY_PHRASES or compact.replace(" ", "") in {
        "ندارد", "ندارم", "نداشت", "نداشتم", "نداردندارد", "هیچ", "هیچی"}
    has_pos = any(h in compact for h in _POSITIVE_EVIDENCE_HINTS)
    has_strong_neg = any(h in compact for h in _STRONG_NEGATIVE_EVIDENCE_HINTS)
    rate = row.get("rate")
    status = str(row.get("recommendation_status", ""))
    metadata_positive = status == "recommended" and rate is not None and pd.notna(rate) and float(rate) >= 4.0
    metadata_negative = status == "not_recommended" or (rate is not None and pd.notna(rate) and float(rate) <= 2.5)
    return {"text": compact, "placeholder": bool(placeholder), "has_positive_language": bool(has_pos),
            "has_strong_negative_language": bool(has_strong_neg), "metadata_positive": bool(metadata_positive),
            "metadata_negative": bool(metadata_negative), "evidence_type": str(row.get("evidence_type", ""))}


def _accept_positive_evidence(row: dict) -> bool:
    f = _procon_flags(row)
    if not f["text"] or f["placeholder"]:
        return False
    if f["has_strong_negative_language"] and not f["has_positive_language"]:
        return False                                  # never label a negative sentence as a strength
    if f["evidence_type"] == "advantage":
        return bool(f["has_positive_language"] or f["metadata_positive"] or not f["has_strong_negative_language"])
    return bool(f["has_positive_language"] or (f["metadata_positive"] and not f["has_strong_negative_language"]))


def _accept_negative_evidence(row: dict) -> bool:
    f = _procon_flags(row)
    if not f["text"] or f["placeholder"]:
        return False
    if f["has_positive_language"] and not f["has_strong_negative_language"]:
        return False                                  # never let a positive review appear under "weaknesses"
    if f["evidence_type"] == "retrieved_review":
        return bool(f["has_strong_negative_language"] or (f["metadata_negative"] and not f["has_positive_language"]))
    if f["evidence_type"] == "disadvantage":
        return bool(not f["has_positive_language"] and not f["placeholder"])
    return bool(f["has_strong_negative_language"] or (f["metadata_negative"] and not f["has_positive_language"]))


def _dedupe_rows(rows, k=3):
    out, seen = [], set()
    for row in rows:
        cid = int(row["comment_id"])
        if cid in seen:
            continue
        seen.add(cid)
        out.append(row)
        if len(out) >= k:
            break
    return out


# ---- citation verification + answer container ---------------------------
_CITE_P = re.compile(r"\[محصول\s*(\d+)\]")
_CITE_R = re.compile(r"\[بازبینی\s*(\d+)\]")
_CITE_ANY = re.compile(r"\[(?:محصول|بازبینی)\s*\d+\]")
_MISSING = ("اطلاعات کافی موجود نیست", "موجود نیست")
_TRAILING_CONNECTORS = {"و", "یا", "که", "اما", "با", "از", "به", "در", "برای"}
_UNIVERSAL_CLAIM_RE = re.compile(
    r"(?:همه|تمام)\s*(?:ی|ٔ)?\s*(?:بازبینی|بازبینی‌ها|نظر|نظرها)|هیچ\s*(?:بازبینی|نظر)")


def verify_citations(text, allowed_products, allowed_reviews) -> str:
    text = _CITE_P.sub(lambda m: m.group(0) if int(m.group(1)) in allowed_products else "", text)
    text = _CITE_R.sub(lambda m: m.group(0) if int(m.group(1)) in allowed_reviews else "", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


@dataclass
class Answer:
    intent: str
    query: str
    text: str
    citations: list = field(default_factory=list)
    review_citations: list = field(default_factory=list)
    sources: list = field(default_factory=list)
    missing_info: bool = False
    tier: str = "extractive"
    latency_s: float = 0.0
    cost_usd: float = 0.0
    needs_clarification: bool = False


class ShoppingAssistant:
    def __init__(self, catalog: Catalog, product_index, review_retriever, llm=None, final_k: int = 8):
        self.c = catalog
        self.pidx = product_index
        self.rrev = review_retriever
        self.llm = llm or LLM(mode="extractive")
        self.router = IntentRouter(catalog)
        self.final_k = final_k
        self.generated_rejections = 0                  # how often the LLM tier failed a quality guard

    def answer(self, query) -> Answer:
        t0 = time.time()
        route = self.router.route(query)
        if route.needs_clarification:
            a = Answer(route.intent, query, "برای مقایسه، دو محصول را با شناسه مشخص کنید.",
                       needs_clarification=True)
        elif route.intent == "discovery":
            a = self._discover(query, route.filters)
        elif route.intent == "product_qa":
            a = self._qa(query, route.product_id)
        elif route.intent == "comparison":
            a = self._compare(query, route.product_ids)
        else:
            a = self._managerial(query, route.scope)
        a.latency_s = round(time.time() - t0, 3)
        a.cost_usd = getattr(self.llm, "last_cost_usd", 0.0)
        return a

    def _gen(self, system, user, extractive, allowed_p, allowed_r):
        """Generate with the LLM tier, or fall back to the deterministic extractive
        answer if the reply is empty, hollow (citation-only), missing a required
        citation, truncated mid-sentence, or asserts an absolute claim ("all
        reviews...") -- a real failure mode caught in testing where the model
        claimed every cited review rated >=3 while one cited review was a 0."""
        g = self.llm.generate(system, user)
        if not g or not g.strip():
            return extractive, "extractive"
        clean = verify_citations(g, allowed_p, allowed_r)

        without_cites = _CITE_ANY.sub(" ", clean)
        meaningful_tokens = [t for t in pt.tokenize(without_cites) if len(t) > 1]
        if len(meaningful_tokens) < 8:
            self.generated_rejections += 1
            return extractive, "extractive"

        if (allowed_p or allowed_r) and not _CITE_ANY.search(clean):
            self.generated_rejections += 1
            return extractive, "extractive"

        tail = pt.normalize(clean).rstrip()
        tail_tokens = pt.tokenize(tail)
        last_token = tail_tokens[-1] if tail_tokens else ""
        if (tail and tail[-1] in "،؛,:-") or last_token in _TRAILING_CONNECTORS:
            self.generated_rejections += 1            # looks token-truncated mid-sentence
            return extractive, "extractive"

        if _UNIVERSAL_CLAIM_RE.search(tail):
            self.generated_rejections += 1             # absolute claim, risk of contradicting evidence
            return extractive, "extractive"

        return clean, "llm"

    def _discover(self, query, filters):
        hits = self.pidx.search(query, filters, k=max(40, self.final_k * 5))
        if not hits:
            return Answer("discovery", query,
                          "هیچ محصولی با این فیلترها یافت نشد (اطلاعات کافی موجود نیست).", missing_info=True)

        qn = pt.normalize(query)
        cheap_pref = any(w in qn for w in ("اقتصادی", "ارزان", "ارزون", "قیمت مناسب", "مقرون به صرفه"))
        satisfaction_pref = any(w in qn for w in ("رضایت", "راضی", "توصیه کاربران", "پیشنهاد کاربران",
                                                  "نظر کاربران خوب", "خریداران راضی"))
        quality_pref = any(w in qn for w in ("باکیفیت", "کیفیت خوب", "بهترین", "خوب")) or satisfaction_pref

        # Retrieval finds relevant candidates; when the request mentions price,
        # quality, or user satisfaction, rerank with those as real data signals
        # (price/rating/review-recommend-rate) instead of leaving it to text match.
        if cheap_pref or quality_pref:
            retrieval_score = _minmax(np.array([h["score"] for h in hits], dtype=float))

            prices = np.array([float(h["price"]) if h["price"] not in (None, 0) else np.nan for h in hits])
            price_score = np.full(len(hits), 0.5, dtype=float)
            good_price = np.isfinite(prices)
            if good_price.sum() >= 2:
                price_score[good_price] = 1.0 - _minmax(np.log1p(prices[good_price]))

            rates = np.array([float(h["rate"]) if h["rate"] is not None else np.nan for h in hits])
            quality_score = np.full(len(hits), 0.5, dtype=float)
            good_rate = np.isfinite(rates)
            if good_rate.sum() >= 2:
                quality_score[good_rate] = _minmax(rates[good_rate])

            counts = np.log1p(np.array([h.get("rate_count", 0) for h in hits], dtype=float))
            confidence = _minmax(counts)

            review_rec_rates, review_support = [], []
            for h in hits:
                stats = review_stats(self.c, int(h["product_id"]))
                rec = stats.get("rec_rate")
                n_lab = int(stats.get("n_labeled_recommendation", 0) or 0)
                review_rec_rates.append(np.nan if rec is None else float(rec))
                review_support.append(n_lab)
                h["review_rec_rate"] = rec
                h["review_labeled_count"] = n_lab
            review_rec_rates = np.array(review_rec_rates, dtype=float)
            satisfaction_score = np.full(len(hits), 0.5, dtype=float)
            good_rec = np.isfinite(review_rec_rates)
            if good_rec.sum() >= 2:
                satisfaction_score[good_rec] = _minmax(review_rec_rates[good_rec])
            review_confidence = _minmax(np.log1p(np.array(review_support, dtype=float)))

            if satisfaction_pref:
                final_score = 0.60 * retrieval_score + 0.18 * satisfaction_score + 0.07 * review_confidence
                if cheap_pref:
                    final_score += 0.08 * price_score
                if quality_pref:
                    final_score += 0.05 * quality_score
                final_score += 0.02 * confidence
            else:
                final_score = 0.72 * retrieval_score
                if cheap_pref and quality_pref:
                    final_score += 0.13 * price_score + 0.10 * quality_score + 0.05 * confidence
                elif cheap_pref:
                    final_score += 0.23 * price_score + 0.05 * confidence
                else:
                    final_score += 0.23 * quality_score + 0.05 * confidence

            order = np.argsort(-final_score, kind="stable")
            hits = [hits[int(i)] for i in order]
            for rank, h in enumerate(hits, 1):
                h["rank"] = rank
                reasons = []
                if cheap_pref and h["price"] is not None:
                    reasons.append("قیمت مناسب‌تر در میان نامزدهای بازیابی‌شده")
                if quality_pref and h["rate"] is not None:
                    reasons.append(f"امتیاز {h['rate']} با {h.get('rate_count', 0)} رأی")
                if satisfaction_pref:
                    rec, n_lab = h.get("review_rec_rate"), int(h.get("review_labeled_count", 0) or 0)
                    reasons.append(f"{rec:.0%} توصیه در {n_lab} بازبینی دارای وضعیت معتبر" if rec is not None and n_lab
                                   else "شواهد وضعیت پیشنهاد کاربران در نمونه کافی نیست")
                h["preference_reason"] = "؛ ".join(reasons)

        ev = hits[:self.final_k]
        allowed = {h["product_id"] for h in ev}
        lines = []
        for h in ev:
            reason = h.get("preference_reason", "")
            reason_text = f"؛ دلیل: {reason}" if reason else ""
            lines.append(f"{h['rank']}. [محصول {h['product_id']}] {h['title']} — امتیاز "
                        f"{h['rate'] if h['rate'] is not None else 'نامشخص'}، قیمت {pt.format_toman(h['price'])} تومان،"
                        f" برند {h['brand'] or 'نامشخص'}{reason_text}")
        extractive = "پیشنهادهای برتر:\n" + "\n".join(lines)

        text, tier = self._gen(prompts.DISCOVERY_SYSTEM.format(k=len(ev)),
                               f"درخواست: {query}\nفیلترها: {filters}\n\nمدارک:\n"
                               f"{prompts.evidence_products(ev)}\n\nپاسخ:",
                               extractive, allowed, set())
        return Answer("discovery", query, text, citations=list(allowed), sources=ev, tier=tier,
                      missing_info=any(p in text for p in _MISSING))

    def _qa(self, query, pid):
        if pid is None:
            return Answer("product_qa", query,
                          "محصول شناسایی نشد؛ شناسه یا نام کامل را ذکر کنید.", missing_info=True)
        prod = self.c.product(pid) or {}
        title = prod.get("title_fa", "")
        candidates = self.rrev.retrieve(query, pid, k=max(20, self.final_k * 3))
        polarity = self.rrev._polarity(query)

        def _flags(row):
            txt = pt.normalize(row.get("text", ""))
            has_neg = any(c in txt for c in self.rrev._NEGATIVE_CUES) or row.get("has_disadvantage", False)
            has_pos = any(c in txt for c in self.rrev._POSITIVE_CUES) or row.get("has_advantage", False)
            return has_pos, has_neg

        if polarity == "negative":
            hits = [h for h in candidates if _flags(h)[1]
                   or (h.get("recommendation_status") == "not_recommended"
                       and h.get("rate") is not None and float(h["rate"]) <= 3.0)][:self.final_k]
        elif polarity == "positive":
            hits = []
            for h in candidates:
                has_pos, has_neg = _flags(h)
                metadata_positive = (h.get("recommendation_status") == "recommended"
                                     and h.get("rate") is not None and float(h["rate"]) >= 4.0)
                if (has_pos or metadata_positive) and not has_neg:  # never reuse a negative review as positive evidence
                    hits.append(h)
            hits = hits[:self.final_k]
        else:
            hits = candidates[:self.final_k]

        if not hits:
            if candidates and polarity in ("negative", "positive"):
                direction = "منفی" if polarity == "negative" else "مثبت"
                return Answer("product_qa", query,
                              f"برای [محصول {pid}] بازبینی مرتبط وجود دارد، اما شواهد {direction} کافی برای پاسخ "
                              f"مطمئن پیدا نشد (اطلاعات کافی موجود نیست).",
                              citations=[pid], sources=candidates[:self.final_k], missing_info=True)
            return Answer("product_qa", query,
                          f"برای [محصول {pid}] بازبینی‌ای نیست (اطلاعات کافی موجود نیست).",
                          citations=[pid], missing_info=True)

        allowed_r = {h["comment_id"] for h in hits}
        facts = review_stats(self.c, pid)
        if facts["n_reviews"] and facts.get("n_labeled_recommendation"):
            head = (f"دربارهٔ [محصول {pid}] ({title}): {facts['n_reviews']} بازبینی در نمونه داریم؛ "
                    f"از {facts['n_labeled_recommendation']} بازبینی دارای وضعیت پیشنهاد، "
                    f"{facts['rec_rate']:.0%} توصیه، {facts['not_rec_rate']:.0%} عدم توصیه و "
                    f"{facts['no_idea_rate']:.0%} بدون نظر قطعی بوده‌اند. [محصول {pid}]")
        elif facts["n_reviews"]:
            head = (f"دربارهٔ [محصول {pid}] ({title}): {facts['n_reviews']} بازبینی در نمونه داریم، اما وضعیت "
                    f"پیشنهاد معتبر کافی ثبت نشده است. [محصول {pid}]")
        else:
            head = f"[محصول {pid}] ({title}):"

        section = ("ایرادها، مشکلات و نکات منفی مرتبط:" if polarity == "negative" else
                  "نقاط قوت و تجربه‌های مثبت مرتبط:" if polarity == "positive" else "بازبینی‌های مرتبط با پرسش:")
        extractive = head + "\n" + section + "\n" + "\n".join(
            f"- «{h['text']}» [بازبینی {h['comment_id']}]" for h in hits)

        text, tier = self._gen(prompts.QA_SYSTEM.format(max_lines=8),
                               f"محصول: [محصول {pid}] {title}\nپرسش: {query}\n\nمدارک:\n"
                               f"{prompts.evidence_reviews(hits)}\n\nپاسخ:",
                               extractive, {pid}, allowed_r)
        return Answer("product_qa", query, text, citations=[pid], review_citations=list(allowed_r),
                      sources=hits, tier=tier, missing_info=any(p in text for p in _MISSING))

    def _compare(self, query, pids):
        facts, positive_by_product, negative_by_product, all_review_rows = [], {}, {}, []

        def _review_polarity(row):
            text_n = pt.normalize(row.get("text", ""))
            rate, status = row.get("rate"), row.get("recommendation_status")
            has_neg = (any(c in text_n for c in self.rrev._NEGATIVE_CUES) or row.get("has_disadvantage", False)
                      or status == "not_recommended" or (rate is not None and pd.notna(rate) and float(rate) <= 2.5))
            has_pos = (any(c in text_n for c in self.rrev._POSITIVE_CUES) or row.get("has_advantage", False)
                      or (status == "recommended" and rate is not None and pd.notna(rate) and float(rate) >= 4.0))
            return "negative" if has_neg else ("positive" if has_pos else "neutral")

        for pid in pids:
            p = self.c.product(pid) or {}
            stats = review_stats(self.c, pid)
            facts.append({"product_id": int(pid), "title": p.get("title_fa", "نامشخص"),
                          "price": p.get("price_clean"), "rate": p.get("product_rate_clean"),
                          "brand": p.get("brand_norm", ""), "stats": stats})

            positives = [dict(r, evidence_type="advantage") for r in stats.get("top_advantages", [])]
            negatives = [dict(r, evidence_type="disadvantage") for r in stats.get("top_disadvantages", [])]
            for row in self.rrev.retrieve(query, pid, k=max(6, min(10, self.final_k + 2))):
                pol = _review_polarity(row)
                if pol == "negative":
                    negatives.append(dict(row, evidence_type="retrieved_review"))
                elif pol == "positive":
                    positives.append(dict(row, evidence_type="retrieved_review"))

            positives = _dedupe_rows([r for r in positives if _accept_positive_evidence(r)], k=3)
            negatives = _dedupe_rows([r for r in negatives if _accept_negative_evidence(r)], k=3)
            positive_by_product[int(pid)], negative_by_product[int(pid)] = positives, negatives
            all_review_rows += positives + negatives

        review_rows, seen = [], set()
        for row in all_review_rows:
            cid = int(row["comment_id"])
            if cid not in seen:
                seen.add(cid)
                review_rows.append(row)

        allowed_p = {f["product_id"] for f in facts}
        allowed_r = {r["comment_id"] for r in review_rows}

        fact_lines = ["۱) واقعیت‌های مستقیم محصول"]
        for f in facts:
            rec = f["stats"].get("rec_rate")
            fact_lines.append(f"- [محصول {f['product_id']}] {f['title']} | برند: {f['brand'] or 'نامشخص'} | "
                              f"قیمت: {pt.format_toman(f['price'])} تومان | امتیاز محصول: "
                              f"{f['rate'] if f['rate'] is not None else 'نامشخص'} | "
                              f"تعداد بازبینی: {f['stats']['n_reviews']} | "
                              f"نرخ توصیه: {f'{rec:.0%}' if rec is not None else 'نامشخص'} | "
                              f"بازبینی‌های دارای نشانهٔ شکایت: {int(f['stats'].get('complaint_count', 0) or 0)}")

        evidence_lines = ["۲) شواهد بازبینی‌های کاربران"]
        for f in facts:
            pid = f["product_id"]
            evidence_lines.append(f"- [محصول {pid}] نقاط قوت / تجربه‌های مثبت:")
            pos = positive_by_product.get(pid, [])
            evidence_lines += [f"  - «{r['text']}» [بازبینی {r['comment_id']}]" for r in pos] or \
                ["  - شواهد مثبت کافی در نمونه پیدا نشد."]
            evidence_lines.append(f"- [محصول {pid}] نقاط ضعف / تجربه‌های منفی:")
            neg = negative_by_product.get(pid, [])
            evidence_lines += [f"  - «{r['text']}» [بازبینی {r['comment_id']}]" for r in neg] or \
                ["  - نقطهٔ ضعف قابل اتکای کافی در نمونه پیدا نشد."]

        inference_lines = ["۳) جمع‌بندی / استنباط از داده‌های بالا"]
        rec_candidates = [f for f in facts if f["stats"].get("rec_rate") is not None]
        if rec_candidates:
            best = max(f["stats"]["rec_rate"] for f in rec_candidates)
            winners = [f for f in rec_candidates if abs(f["stats"]["rec_rate"] - best) < 1e-12]
            if len(winners) == 1:
                w = winners[0]
                inference_lines.append(f"- اگر رضایت کاربران اولویت اصلی باشد، [محصول {w['product_id']}] در "
                                       f"دادهٔ فعلی نرخ توصیهٔ بالاتری دارد ({w['stats']['rec_rate']:.0%}).")
            else:
                tied = " و ".join(f"[محصول {f['product_id']}]" for f in winners)
                inference_lines.append(f"- از نظر نرخ توصیه، {tied} در دادهٔ فعلی برابرند ({best:.0%})؛ شواهد "
                                       f"قوت/ضعف و قیمت برای تصمیم نهایی مهم می‌شوند.")
        priced = [f for f in facts if f["price"] is not None and pd.notna(f["price"]) and float(f["price"]) > 0]
        if priced:
            cheapest = min(priced, key=lambda x: float(x["price"]))
            inference_lines.append(f"- اگر قیمت اولویت اصلی باشد، [محصول {cheapest['product_id']}] با قیمت "
                                   f"{pt.format_toman(cheapest['price'])} تومان ارزان‌تر است.")
        complaint_candidates = [f for f in facts if int(f["stats"].get("n_reviews", 0) or 0) > 0]
        if len(complaint_candidates) >= 2:
            rates = sorted(((int(f["stats"].get("complaint_count", 0) or 0)
                            / max(1, int(f["stats"].get("n_reviews", 0) or 0)), f) for f in complaint_candidates),
                           key=lambda x: x[0])
            low_rate, low_item = rates[0]
            high_rate = rates[-1][0]
            if high_rate - low_rate >= 0.10:
                inference_lines.append(f"- در نمونهٔ فعلی، [محصول {low_item['product_id']}] سهم کمتری از "
                                       f"بازبینی‌های دارای نشانهٔ شکایت دارد ({low_rate:.0%}).")
        if len(inference_lines) == 1:
            inference_lines.append("- برای نتیجه‌گیری قطعی اطلاعات کافی موجود نیست.")

        # Comparison stays fully deterministic: every fact and inference above is
        # computed directly from selected data, not phrased by an LLM.
        text = "\n".join(fact_lines + [""] + evidence_lines + [""] + inference_lines)
        return Answer("comparison", query, text, citations=list(allowed_p), review_citations=list(allowed_r),
                      sources={"facts": facts, "positive_reviews": positive_by_product,
                              "negative_reviews": negative_by_product},
                      tier="extractive", missing_info=any(p in text for p in _MISSING))

    def _managerial(self, query, scope):
        summary = managerial_summary(self.c, scope)
        if not summary.get("n_products"):
            return Answer("managerial", query,
                          f"برای دامنهٔ {scope} داده‌ای نیست (اطلاعات کافی موجود نیست).", missing_info=True)
        terms = " ".join(t["term"] for t in summary.get("top_complaint_terms", [])[:3])
        low_pids = [int(p["product_id"]) for p in summary.get("low_recommendation_products", [])]
        pids = low_pids + [int(p) for p in summary.get("product_ids", []) if int(p) not in set(low_pids)]

        def _is_negative_complaint(row):
            text_n = pt.normalize(row.get("text", ""))
            rate = row.get("rate")
            low_rate = rate is not None and pd.notna(rate) and float(rate) <= 2.5
            return bool(any(c in text_n for c in self.rrev._NEGATIVE_CUES) or row.get("has_disadvantage", False)
                       or row.get("recommendation_status") == "not_recommended" or low_rate)

        complaint_reviews, seen = [], set()
        if terms and pids:
            for pid in pids[:20]:
                for r in self.rrev.retrieve(terms, pid, k=2):
                    cid = int(r["comment_id"])
                    if cid in seen or not _is_negative_complaint(r):
                        continue
                    seen.add(cid)
                    complaint_reviews.append(r)
        complaint_reviews = complaint_reviews[:8]

        allowed_p = {p["product_id"] for p in summary["low_recommendation_products"]}
        allowed_r = {r["comment_id"] for r in complaint_reviews}
        weighted_rec = summary.get("review_weighted_rec_rate")
        threshold = summary.get("low_recommendation_min_reviews")

        ev = [f"دامنه: {scope.get('value')}",
             f"تعداد محصول دارای بازبینی: {summary['n_products']} | تعداد بازبینی: {summary['n_reviews']} | "
             f"وضعیت پیشنهاد معتبر: {summary.get('n_labeled_recommendations', 0)} | "
             f"میانگین امتیاز بازبینی: {summary['avg_rate']} | "
             f"نرخ توصیهٔ وزن‌دار در بین وضعیت‌های معتبر: {f'{weighted_rec:.1%}' if weighted_rec is not None else 'نامشخص'}",
             "شکایت‌های پرتکرار: " + (", ".join(f"{t['term']}({t['count']})"
                                                for t in summary["top_complaint_terms"]) or "نامشخص")]
        brands = summary.get("brand_satisfaction") or []
        if brands:
            ev.append("الگوی رضایت برندها:")
            ev += [f"- {b['brand']} | {b['n_products']} محصول | {b['n_reviews']} بازبینی | "
                  f"نرخ توصیهٔ وزن‌دار {b['review_weighted_rec_rate']:.1%}" for b in brands[:5]]
        ev.append(f"محصولات با نرخ توصیهٔ پایین (حداقل {threshold} بازبینی در نمونه):" if threshold is not None
                  else "محصولات با نرخ توصیهٔ پایین: دادهٔ کافی برای آستانهٔ حداقل دو بازبینی موجود نیست.")
        ev += [f"- [محصول {p['product_id']}] {p['title']} | نرخ توصیه {p['rec_rate']:.1%} | {p['n_reviews']} بازبینی"
              for p in summary["low_recommendation_products"]]
        evidence_text = "\n".join(ev)
        extractive = evidence_text + ("\n\nنمونهٔ شکایت‌های قابل ردیابی:\n" + "\n".join(
            f"- «{r['text']}» [بازبینی {r['comment_id']}]" for r in complaint_reviews) if complaint_reviews else "")

        # Deterministic by design: prevents unsupported generalizations about
        # price/quality while cutting API cost and latency for analytics queries.
        return Answer("managerial", query, extractive, citations=list(allowed_p), review_citations=list(allowed_r),
                      sources=[summary], tier="extractive",
                      missing_info=any(p in extractive for p in _MISSING))


# ---- non-LLM lexical baseline (evaluation control) ----------------------
class LexicalBaseline:
    """Control: token-overlap product retrieval + arithmetic, no embeddings, no LLM."""

    def __init__(self, catalog: Catalog):
        self.c = catalog

    def discover(self, query, k: int = 5):
        f = extract_filters(self.c, query)
        mask = product_filter_mask(self.c.products, f)
        sub = self.c.products[mask]
        q = set(pt.tokenize(query))
        sub = sub.assign(_ov=sub["product_text_norm"].map(lambda t: len(q & set(pt.tokenize(t)))))
        sub = sub.sort_values(["_ov", "product_rate_clean", "rate_count"], ascending=False).head(k)
        return sub[["product_id", "title_fa", "price_clean", "product_rate_clean"]]


def build_assistant(llm=None, final_k: int = 8) -> ShoppingAssistant:
    """Wire a ready-to-use assistant from the cached index + cleaned comments."""
    idx = retrieval.ProductIndex.load()
    by_product = retrieval.load_comments_by_product()
    catalog = Catalog.build(idx.products, by_product)
    rrev = retrieval.ReviewRetriever(by_product)
    return ShoppingAssistant(catalog, idx, rrev, llm=llm, final_k=final_k)

### 2.6 Build the assistant + demo the four capabilities
This is exactly what `digikala.demo.build_assistant` (and the packaged app) do, so the notebook and the application produce identical results on the same sample.

In [14]:
products_df = _prepare_products(products_df, comments_df)      # add comment_count
pidx = ProductIndex.build(products_df)
by_product = GroupedComments(comments_df[comments_df["has_text"].astype(bool)])
catalog = Catalog.build(pidx.products, by_product)
assistant = ShoppingAssistant(catalog, pidx, ReviewRetriever(by_product), llm=LLM(mode=RUN_MODE))
print("assistant ready | products", len(catalog.products), "| reviewed", len(by_product))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

assistant ready | products 17043 | reviewed 17043


In [15]:
def show(ans):
    print(f"intent={ans.intent} tier={ans.tier} latency={ans.latency_s}s "
          f"cites={ans.citations[:3]} rev={ans.review_citations[:3]}")
    print(ans.text[:400], "\n")

top = catalog.products.sort_values("comment_count", ascending=False)["product_id"].head(2).astype(int).tolist()
cat = catalog.products["category1_norm"].replace("نامشخص", np.nan).dropna().value_counts().index[0]
show(assistant.answer("یک کالای اقتصادی و باکیفیت زیر ۵۰۰ هزار تومان می‌خواهم"))       # discovery
show(assistant.answer(f"آیا کاربران از کیفیت محصول {top[0]} راضی بودند؟"))              # Q&A
show(assistant.answer(f"محصول {top[0]} و محصول {top[1]} را مقایسه کن"))                 # comparison
show(assistant.answer(f"پرتکرارترین شکایت‌ها در دستهٔ {cat} چیست؟"))                    # managerial

intent=discovery tier=extractive latency=0.164s cites=[822241, 3135078, 170694] rev=[]
پیشنهادهای برتر:
1. [محصول 3135078] ساختنی سلام بازی مدل میله هزار کاره کد 500 — امتیاز 92.0، قیمت 235,000 تومان، برند متفرقه؛ دلیل: قیمت مناسب‌تر در میان نامزدهای بازیابی‌شده؛ امتیاز 92.0 با 175 رأی
2. [محصول 822241] زیر نشیمنی مدل نوین — امتیاز 74.0، قیمت 60,000 تومان، برند متفرقه؛ دلیل: قیمت مناسب‌تر در میان نامزدهای بازیابی‌شده؛ امتیاز 74.0 با 99 رأی
3. [محصول 7523408] سویه هزار گیاه نوند - 30 



intent=product_qa tier=extractive latency=0.053s cites=[2487917] rev=[48820256, 48730211, 51432548]
دربارهٔ [محصول 2487917] (واکس مو حالت دهنده دور لایت کد 001 حجم 300 میلی لیتر): 6 بازبینی در نمونه داریم؛ از 5 بازبینی دارای وضعیت پیشنهاد، 80% توصیه، 0% عدم توصیه و 20% بدون نظر قطعی بوده‌اند. [محصول 2487917]
نقاط قوت و تجربه‌های مثبت مرتبط:
- «واکس دورلایت خیلی خوبه. کیفیت واقعا قابل قبولی داره نسبت به قیمتش. از تمام هم رده هاش بهتره.» [بازبینی 48730211]
- «واکس مو خوب بود بوی خوبی دارد» [بازبین 

intent=comparison tier=extractive latency=0.075s cites=[170516, 2487917] rev=[27769187, 51432548, 9440933]
۱) واقعیت‌های مستقیم محصول
- [محصول 2487917] واکس مو حالت دهنده دور لایت کد 001 حجم 300 میلی لیتر | برند: دور لایت | قیمت: 45,000 تومان | امتیاز محصول: 82 | تعداد بازبینی: 6 | نرخ توصیه: 80% | بازبینی‌های دارای نشانهٔ شکایت: 0
- [محصول 170516] پشت کمری هوشمند مدل Office | برند: هوشمند | قیمت: 479,000 تومان | امتیاز محصول: 80 | تعداد بازبینی: 6 | نرخ توصیه: 100% | بازبینی‌های دارای نشانهٔ

intent=managerial tier=extractive latency=4.107s cites=[1516000, 1025606, 7429067] rev=[21468040, 48060585, 25544488]
دامنه: اسباب بازی
تعداد محصول دارای بازبینی: 1130 | تعداد بازبینی: 1320 | وضعیت پیشنهاد معتبر: 1093 | میانگین امتیاز بازبینی: 3.7 | نرخ توصیهٔ وزن‌دار در بین وضعیت‌های معتبر: 75.7%
شکایت‌های پرتکرار: بی کیفیت(16), پاره(13), خراب(13), افتضاح(5), مشکل(3), شکسته(2), بوی(2), خش(2)
الگوی رضایت برندها:
- اسلایم تپلی | 18 محصول | 23 بازبینی | نرخ توصیهٔ وزن‌دار 76.2%
- دنیای سرگرمی های کمیاب | 17 محصول | 



## 3 · Phase 3 — Recommendation-status prediction (Macro-F1)

**Text-only** model (TF-IDF + linear). We deliberately exclude `rate_clean`, `likes`
and `is_buyer`: `rate` is a second copy of the same sentiment as the label (leakage),
`likes` accrue after posting (temporal leakage), and the brief asks us to predict from
the *text*. We still report a text-vs-text+numeric ablation to quantify that leak.

**Primary metric is the product-grouped Macro-F1**, not the naive random split: a
random split can still put two different reviews of the *same* product in both train
and test (brand names, model-specific phrasing) even after exact-duplicate-text dedup.
The grouped split holds whole products out, and we also report exactly how much
product overlap the naive split has (`naive_split_product_overlap_pct`), so any gap
between the two numbers is explained by data, not asserted away.

In [16]:
import json
import logging

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler


log = logging.getLogger("digikala.phase3")

RANDOM_STATE = config.RANDOM_SEED
FA_TOKEN_PATTERN = r"[؀-ۿ0-9A-Za-z]+"
# Non-textual features. NOT used by the model — kept only for the leakage ablation.
NUMERIC_FEATURES = ["rate_clean", "likes", "is_buyer_num"]
MAX_PER_CLASS = 30_000

PERSIAN_STOPWORDS = [
    "و", "در", "به", "از", "که", "این", "با", "را", "برای", "رو", "هم", "یک", "ها",
    "است", "نیز", "شد", "شود", "می", "خواهد", "بر", "آن", "تا", "کرد", "دارد", "بود",
    "اما", "اگر", "هر", "همه", "خیلی", "بیشتر", "کمتر", "مثل", "مانند", "حتی",
]


def _prep(df: pd.DataFrame) -> pd.DataFrame:
    """Keep labeled, text-bearing, product-matched rows; dedup by text; fill numerics.
    Shared by the full-data loader and the in-memory demo path so both agree."""
    mask = (df["recommendation_valid"].fillna(False) & df["has_text"].fillna(False)
            & df["product_id"].notna())
    df = df.loc[mask].drop_duplicates(subset="comment_text_norm", keep="first").copy()
    df["is_buyer_num"] = df["is_buyer"].fillna(False).astype(int)
    df["rate_clean"] = df["rate_clean"].fillna(0)
    df["likes"] = df["likes"].fillna(0)
    return df


def _load() -> pd.DataFrame:
    cols = ["comment_id", "product_id", "comment_text_norm", "recommendation_status",
            "recommendation_valid", "has_text", "rate_clean", "likes", "is_buyer"]
    return _prep(pd.read_parquet(config.COMMENTS_CLEAN, columns=cols))


def _stratified_cap(df: pd.DataFrame, col: str, cap: int) -> pd.DataFrame:
    parts = [g.sample(n=min(len(g), cap), random_state=RANDOM_STATE)
             for _, g in df.groupby(col, sort=False)]
    return pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


def _vectorizer(max_features: int = 50_000) -> TfidfVectorizer:
    return TfidfVectorizer(token_pattern=FA_TOKEN_PATTERN, ngram_range=(1, 2),
                           min_df=5, max_df=0.8, sublinear_tf=True,
                           max_features=max_features, stop_words=PERSIAN_STOPWORDS)


def _pipeline(clf, numeric: bool = False) -> Pipeline:
    """Text-only by default. numeric=True adds the leaky metadata (ablation only)."""
    trans = [("text", _vectorizer(), "comment_text_norm")]
    if numeric:
        trans.append(("num", StandardScaler(), NUMERIC_FEATURES))
    return Pipeline([("preprocessor", ColumnTransformer(trans)), ("clf", clf)])


def _xy(df: pd.DataFrame, numeric: bool = False):
    cols = ["comment_text_norm"] + (NUMERIC_FEATURES if numeric else [])
    return df[cols], df["target_encoded"]


def _logreg():
    return LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0,
                              solver="saga", random_state=RANDOM_STATE)


def prepare_split(df: pd.DataFrame):
    """Shared label-encode + stratified-cap + product-grouped split. Deterministic
    (fixed RANDOM_STATE) so this reproduces the exact same split _train() uses
    internally -- used by the LoRA fine-tune script so it's compared against the
    TF-IDF baseline on the identical held-out product-grouped test set."""
    le = LabelEncoder()
    df = df.copy()
    df["target_encoded"] = le.fit_transform(df["recommendation_status"])
    sample = _stratified_cap(df, "target_encoded", MAX_PER_CLASS)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    gtr, gte = next(gss.split(sample, groups=sample["product_id"]))
    g_train, g_test = sample.iloc[gtr].copy(), sample.iloc[gte].copy()
    assert not (set(g_train["product_id"]) & set(g_test["product_id"]))
    return sample, g_train, g_test, le


def _train(df: pd.DataFrame) -> tuple[dict, dict]:
    """Train text-only model + baselines + grouped split + leakage ablation on a
    prepped frame. Returns (model_bundle, metrics). Persists nothing — callers do."""
    log.info("training rows: %d", len(df))
    le = LabelEncoder()
    df["target_encoded"] = le.fit_transform(df["recommendation_status"])
    inv = {i: c for i, c in enumerate(le.classes_)}

    sample = _stratified_cap(df, "target_encoded", MAX_PER_CLASS)
    train_df, temp = train_test_split(sample, test_size=0.40, stratify=sample["target_encoded"],
                                      random_state=RANDOM_STATE)
    val_df, test_df = train_test_split(temp, test_size=0.50, stratify=temp["target_encoded"],
                                       random_state=RANDOM_STATE)
    # leakage guard: no shared comment text across splits
    assert not (set(train_df["comment_text_norm"]) & set(test_df["comment_text_norm"]))

    # quantify the residual leakage risk of the naive random split: how many test
    # products were *also* seen (via a different review) in training?
    train_pids, test_pids = set(train_df["product_id"]), set(test_df["product_id"])
    naive_overlap_pct = round(100 * len(train_pids & test_pids) / max(1, len(test_pids)), 2)

    # ---- text-only (the real model) ----
    X_train, y_train = _xy(train_df)
    X_val, y_val = _xy(val_df)
    X_test, y_test = _xy(test_df)

    # baselines (majority + logistic regression), text-only, on the validation split
    baselines = {}
    for name, clf in [("majority", DummyClassifier(strategy="most_frequent")),
                      ("logreg", LogisticRegression(max_iter=1000, class_weight="balanced",
                                                    C=1.0, random_state=RANDOM_STATE))]:
        pipe = _pipeline(clf, numeric=False).fit(X_train, y_train)
        baselines[name] = round(f1_score(y_val, pipe.predict(X_val), average="macro"), 4)
    log.info("baselines (val macro-F1): %s", baselines)

    # final model: text-only logistic regression (saga) — cheap, strong, reproducible
    final = _pipeline(_logreg(), numeric=False).fit(X_train, y_train)
    y_pred = final.predict(X_test)
    test_macro_f1 = round(f1_score(y_test, y_pred, average="macro"), 4)
    labels = list(le.classes_)
    y_test_label = [inv[i] for i in y_test]
    y_pred_label = [inv[i] for i in y_pred]
    report = classification_report(y_test_label, y_pred_label,
                                   labels=labels, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_test, y_pred).tolist()

    # failure analysis: which (true, pred) confusions are most common, and a
    # handful of concrete misclassified review texts to inspect
    err = pd.DataFrame({"comment_text": test_df["comment_text_norm"].reset_index(drop=True),
                        "true": y_test_label, "pred": y_pred_label})
    err = err[err["true"] != err["pred"]]
    error_pairs = (err.groupby(["true", "pred"]).size().reset_index(name="count")
                  .sort_values("count", ascending=False).to_dict("records"))
    failure_examples = (err.sample(n=min(12, len(err)), random_state=RANDOM_STATE)
                        .assign(comment_text=lambda x: x["comment_text"].str.slice(0, 350))
                        .to_dict("records") if len(err) else [])

    # product-grouped validation (no product in both train and test), text-only
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
    gtr, gte = next(gss.split(sample, groups=sample["product_id"]))
    g_train, g_test = sample.iloc[gtr], sample.iloc[gte]
    assert not (set(g_train["product_id"]) & set(g_test["product_id"]))
    gpipe = _pipeline(_logreg(), numeric=False).fit(*_xy(g_train))
    grouped_macro_f1 = round(f1_score(g_test["target_encoded"], gpipe.predict(_xy(g_test)[0]),
                                      average="macro"), 4)

    # ---- leakage ablation: text-only vs text + numeric (rate/likes/is_buyer) ----
    numeric_pipe = _pipeline(_logreg(), numeric=True).fit(*_xy(train_df, numeric=True))
    numeric_test_f1 = round(f1_score(y_test, numeric_pipe.predict(_xy(test_df, numeric=True)[0]),
                                     average="macro"), 4)
    ablation = {"text_only_macro_f1": test_macro_f1,
                "text_plus_numeric_macro_f1": numeric_test_f1,
                "leakage_lift": round(numeric_test_f1 - test_macro_f1, 4),
                "note": "The lift from adding rate_clean/likes/is_buyer is leakage: rate "
                        "restates the label, likes are post-hoc. The final model excludes them."}
    log.info("ablation text-only %.4f vs text+numeric %.4f (leak +%.4f)",
             test_macro_f1, numeric_test_f1, ablation["leakage_lift"])

    bundle = {"pipeline": final, "label_encoder": le, "labels": labels,
              "features": ["comment_text_norm"]}
    metrics = {"n_rows": int(len(df)), "n_sampled": int(len(sample)),
               "features_used": ["comment_text_norm"],
               "excluded_to_avoid_leakage": NUMERIC_FEATURES,
               "baselines_val_macro_f1": baselines,
               "primary_macro_f1": grouped_macro_f1,
               "primary_split": "product_grouped",
               "test_macro_f1": test_macro_f1,
               "grouped_macro_f1": grouped_macro_f1,
               "naive_split_product_overlap_pct": naive_overlap_pct,
               "leakage_ablation": ablation,
               "labels": labels, "confusion_matrix": cm, "classification_report": report,
               "error_pairs": error_pairs, "failure_examples": failure_examples}
    log.info("TEXT-ONLY test macro-F1 %.4f | grouped(PRIMARY) %.4f | naive-split product overlap %.2f%%",
             test_macro_f1, grouped_macro_f1, naive_overlap_pct)
    return bundle, metrics


def train_and_save() -> dict:
    """Full-data Phase-3 run: train on the cleaned parquet and persist model+metrics."""
    bundle, metrics = _train(_load())
    model_path = config.MODELS_DIR / "recommendation_model.pkl"
    joblib.dump(bundle, model_path)
    metrics["model_path"] = str(model_path)
    (config.METRICS_DIR / "phase3_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    log.info("saved %s", model_path.name)
    return metrics


def train_from_frame(comments_df: pd.DataFrame) -> tuple[dict, dict]:
    """Train from an in-memory cleaned-comments frame (the shared demo path used by
    the notebook and `run.py demo`). Returns (bundle, metrics); persists nothing so
    the full-data artifacts are never clobbered."""
    return _train(_prep(comments_df))


def fig_confusion(metrics: dict):
    """Plotly confusion-matrix heatmap for the dashboard/notebook."""
    import plotly.express as px
    cm = np.array(metrics["confusion_matrix"])
    labels = metrics["labels"]
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig = px.imshow(cm_norm, x=labels, y=labels, text_auto=".2f", color_continuous_scale="Blues",
                    labels={"x": "predicted", "y": "true", "color": "row-normalized"},
                    title=f"Confusion matrix (product-grouped Macro-F1 = {metrics.get('primary_macro_f1', metrics.get('grouped_macro_f1'))})")
    return fig


def load_model():
    return joblib.load(config.MODELS_DIR / "recommendation_model.pkl")


def predict_with(bundle, texts) -> list[str]:
    """Classify text(s) with a given model bundle (text-only)."""
    if isinstance(texts, str):
        texts = [texts]
    X = pd.DataFrame({"comment_text_norm": [pt.normalize(t) for t in texts]})
    preds = bundle["pipeline"].predict(X)
    return [bundle["label_encoder"].inverse_transform([p])[0] for p in preds]


def predict(texts) -> list[str]:
    """Classify raw review text(s) from TEXT ONLY — used by the dashboard 'Try it!'."""
    return predict_with(load_model(), texts)

In [17]:
# train text-only on the in-notebook cleaned comments (same code as run.py demo)
bundle, p3_metrics = train_from_frame(comments_df)
print("baselines:", p3_metrics["baselines_val_macro_f1"])
print("PRIMARY (product-grouped) Macro-F1:", p3_metrics["primary_macro_f1"],
      "| naive random-split Macro-F1:", p3_metrics["test_macro_f1"],
      "| naive split product overlap:", p3_metrics["naive_split_product_overlap_pct"], "%")
print("leakage ablation:", p3_metrics["leakage_ablation"])
fig_confusion(p3_metrics).show()

E:\QBC13_AI_G6_Project3\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


E:\QBC13_AI_G6_Project3\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


baselines: {'majority': 0.2908, 'logreg': 0.6512}
PRIMARY (product-grouped) Macro-F1: 0.4924 | naive random-split Macro-F1: 0.6103 | naive split product overlap: 14.34 %
leakage ablation: {'text_only_macro_f1': 0.6103, 'text_plus_numeric_macro_f1': 0.6626, 'leakage_lift': 0.0523, 'note': 'The lift from adding rate_clean/likes/is_buyer is leakage: rate restates the label, likes are post-hoc. The final model excludes them.'}


In [18]:
predict_with(bundle, ["کیفیتش عالی بود حتما بخرید", "افتضاح بود پشیمونم"])

['recommended', 'not_recommended']

## 4 · Phase 4 — Evaluation (six axes)
Retrieval quality, grounding, response quality (LLM-judge), prediction, latency, cost, failure analysis — plus a non-LLM baseline control.

In [19]:
import json
import logging
import re
import time

import numpy as np
import pandas as pd

from collections import Counter


log = logging.getLogger("digikala.phase4")
_CITE = re.compile(r"\[(?:محصول|بازبینی)\s*\d+\]")
_CITE_P_EVAL = re.compile(r"\[محصول\s*(\d+)\]")
_CITE_R_EVAL = re.compile(r"\[بازبینی\s*(\d+)\]")
_GENERIC_VALUES = {"", "نامشخص", "سایر", "متفرقه"}


# ---- retrieval metrics --------------------------------------------------
def _dcg(rels):
    return sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(rels))


def ranking_metrics(retrieved_ids, relevant_ids, k=None) -> dict:
    k = k or len(retrieved_ids)
    retrieved = retrieved_ids[:k]
    hits = [1 if i in relevant_ids else 0 for i in retrieved]
    recall = sum(hits) / max(1, len(relevant_ids))
    mrr = next((1.0 / r for r, i in enumerate(retrieved, 1) if i in relevant_ids), 0.0)
    ideal = sorted([1] * min(len(relevant_ids), k), reverse=True)
    ndcg = _dcg(hits) / max(_dcg(ideal), 1e-9)
    return {"recall@k": round(recall, 4), "mrr": round(mrr, 4), "ndcg@k": round(ndcg, 4)}


def evaluate_retrieval(assistant, n_queries: int = 30, k: int = 10) -> dict:
    """Auto-labeled retrieval eval: a product's title is the query, its id the gold.

    Uses the most-reviewed products so the queries are realistic and each has a
    single known-relevant id.
    """
    products = assistant.c.products
    pids = (products.sort_values("comment_count", ascending=False)["product_id"]
            .head(n_queries).astype(int).tolist())
    rows = []
    for pid in pids:
        title = assistant.c.product(pid)["title_fa"]
        hits = assistant.pidx.search(title, k=k)
        rows.append(ranking_metrics([h["product_id"] for h in hits], {pid}, k=k))
    df = pd.DataFrame(rows)
    return {"n_queries": len(rows), "k": k,
            "recall@k": round(df["recall@k"].mean(), 4),
            "mrr": round(df["mrr"].mean(), 4),
            "ndcg@k": round(df["ndcg@k"].mean(), 4)}


def retrieval_ablation(assistant, n_queries: int = 30, k: int = 10) -> dict:
    """Quantify hybrid retrieval vs. its two single-method halves (dense-only,
    BM25-only), using the same title->own-id auto-labels as evaluate_retrieval,
    so the three numbers are directly comparable (bonus: 'Hybrid Search with
    quantified improvement').

    Caveat, reported not hidden: this auto-labeled benchmark uses each product's
    own TITLE as the query, which is a near-exact lexical-match task -- it
    structurally favors BM25 and is not a fair test of hybrid's actual value on
    fuzzy natural-language queries (which the discovery/QA intents actually get).
    On this specific benchmark BM25-only can legitimately beat hybrid; that is
    a real, measured result about the benchmark's bias, not a failure to fix."""
    products = assistant.c.products
    pids = (products.sort_values("comment_count", ascending=False)["product_id"]
            .head(n_queries).astype(int).tolist())
    per_method = {}
    for method in ("dense", "bm25", "hybrid"):
        rows = []
        for pid in pids:
            title = assistant.c.product(pid)["title_fa"]
            hits = assistant.pidx.search(title, k=k, method=method)
            rows.append(ranking_metrics([h["product_id"] for h in hits], {pid}, k=k))
        df = pd.DataFrame(rows)
        per_method[method] = {"recall@k": round(df["recall@k"].mean(), 4),
                               "mrr": round(df["mrr"].mean(), 4),
                               "ndcg@k": round(df["ndcg@k"].mean(), 4)}
    return {"n_queries": len(pids), "k": k, "by_method": per_method,
            "hybrid_vs_best_single_mrr_lift": round(
                per_method["hybrid"]["mrr"] - max(per_method["dense"]["mrr"], per_method["bm25"]["mrr"]), 4)}


# ---- natural-language retrieval benchmark (fixes the title-query bias) --
def build_natural_retrieval_cases(assistant, n_queries: int = 20):
    """Build a diverse Persian pseudo-gold retrieval benchmark using paraphrased
    brand+category+distinguishing-title-token queries, NOT each product's exact
    title (which `evaluate_retrieval`/`retrieval_ablation` use and which is a
    near-exact lexical match that structurally favors BM25). Gold ids are still
    programmatically derived from catalogue metadata -- explicitly not human
    annotation -- but the query text itself is a realistic paraphrase, and at
    most 4 cases come from the same category so one high-volume category can't
    dominate the benchmark."""
    p = assistant.c.products.copy()
    for col in ("product_id", "title_norm", "brand_norm", "category1_norm", "category2_norm", "sub_category_norm"):
        if col not in p:
            p[col] = ""
    p = p[p["product_id"].notna() & p["brand_norm"].notna()
         & (p["brand_norm"].astype(str).str.strip() != "") & (~p["brand_norm"].isin(_GENERIC_VALUES))].copy()
    if "comment_count" in p:
        p = p[p["comment_count"] > 0]
    common = {"مدل", "سری", "اصل", "اورجینال", "جدید", "محصول", "عدد", "بسته", "مناسب", "کیفیت", "خوب", "رنگ", "طرح"}
    candidates, seen_queries, seen_targets = [], set(), set()
    specs = [("brand_norm", "category1_norm"), ("brand_norm", "category2_norm"), ("brand_norm", "sub_category_norm")]
    max_per_category = max(2, min(4, int(np.ceil(n_queries / 5))))
    category_counts = Counter()

    for brand_col, cat_col in specs:
        tmp = p[p[cat_col].notna() & (p[cat_col].astype(str).str.strip() != "")
               & (~p[cat_col].isin(_GENERIC_VALUES))].copy()
        grouped = []
        for (brand, category), g in tmp.groupby([brand_col, cat_col], sort=False):
            if len(g) < 2 or len(g) > 60:
                continue
            strength = int(g.get("comment_count", pd.Series(0, index=g.index)).sum())
            grouped.append((strength, str(brand), str(category), g))
        grouped.sort(key=lambda x: x[0], reverse=True)

        for _, brand, category, g in grouped:
            category_fa = category if re.search(r"[آ-ی]", category) else "کالا"
            if category_counts[category_fa] >= max_per_category:
                continue
            g = g.sort_values(["comment_count", "rate_count"], ascending=False)
            token_df = Counter()
            for title in g["title_norm"].fillna(""):
                token_df.update(set(pt.tokenize_norm(title)))
            forbidden = set(pt.tokenize_norm(brand)) | set(pt.tokenize_norm(category)) | common

            for target in g.head(min(6, len(g))).itertuples():
                target_id = int(target.product_id)
                if target_id in seen_targets:
                    continue
                title_tokens = [t for t in pt.tokenize_norm(str(target.title_norm))
                                if len(t) >= 2 and not t.isdigit() and t not in forbidden and re.search(r"[آ-ی]", t)]
                if not title_tokens:
                    continue
                title_tokens = sorted(dict.fromkeys(title_tokens), key=lambda t: (token_df.get(t, 999999), -len(t)))
                desc = " ".join(title_tokens[:2])
                q = f"یک {category_fa} از برند {brand} می‌خواهم؛ ترجیحاً مدلی که مشخصهٔ «{desc}» را داشته باشد."
                if q in seen_queries:
                    continue
                seen_queries.add(q); seen_targets.add(target_id); category_counts[category_fa] += 1
                candidates.append({"query": q, "relevant_ids": [target_id], "brand": brand,
                                   "category": category, "target_title": str(target.title_norm)})
                break
            if len(candidates) >= n_queries:
                break
        if len(candidates) >= n_queries:
            break
    return candidates[:n_queries]


def evaluate_retrieval_natural(assistant, n_queries: int = 20, k: int = 10) -> dict:
    """Hybrid vs. lexical-baseline retrieval quality on the natural-language
    benchmark above -- this is the fair test of hybrid's value (unlike the
    title-exact-match benchmark, which structurally favors BM25). Reports
    whichever side actually wins, honestly, per query and in aggregate."""
    cases = build_natural_retrieval_cases(assistant, n_queries=n_queries)
    if not cases:
        return {"n_queries": 0, "note": "could not build a natural-language benchmark from the current sample"}
    baseline = LexicalBaseline(assistant.c)
    rows = []
    for case in cases:
        q, relevant = case["query"], set(case["relevant_ids"])
        filters = extract_filters(assistant.c, q)
        hybrid_ids = [int(x["product_id"]) for x in assistant.pidx.search(q, filters=filters, k=k)]
        hm = ranking_metrics(hybrid_ids, relevant, k=k)
        base_ids = [int(x) for x in baseline.discover(q, k=k)["product_id"].tolist()]
        bm = ranking_metrics(base_ids, relevant, k=k)
        rows.append({"query": q, "hybrid_recall@k": hm["recall@k"], "hybrid_mrr": hm["mrr"],
                     "hybrid_ndcg@k": hm["ndcg@k"], "baseline_recall@k": bm["recall@k"],
                     "baseline_mrr": bm["mrr"], "baseline_ndcg@k": bm["ndcg@k"]})
    df = pd.DataFrame(rows)
    hybrid_ndcg, baseline_ndcg = float(df["hybrid_ndcg@k"].mean()), float(df["baseline_ndcg@k"].mean())
    verdict = ("hybrid_better" if hybrid_ndcg > baseline_ndcg + 1e-9
              else "tie" if abs(hybrid_ndcg - baseline_ndcg) <= 1e-9 else "lexical_baseline_better")
    return {"n_queries": len(df), "k": k,
           "benchmark_type": "reproducible_programmatic_pseudo_gold_not_human_labeled",
           "quality_verdict_by_ndcg": verdict,
           "hybrid": {"recall@k": round(float(df["hybrid_recall@k"].mean()), 4),
                     "mrr": round(float(df["hybrid_mrr"].mean()), 4), "ndcg@k": round(hybrid_ndcg, 4)},
           "lexical_baseline": {"recall@k": round(float(df["baseline_recall@k"].mean()), 4),
                               "mrr": round(float(df["baseline_mrr"].mean()), 4), "ndcg@k": round(baseline_ndcg, 4)},
           "per_query": df.to_dict("records")}


# ---- grounding / quality ------------------------------------------------
def citation_coverage(text: str) -> float:
    sents = [s for s in re.split(r"[.!؟\n]+", text) if s.strip()]
    if not sents:
        return 0.0
    return round(sum(1 for s in sents if _CITE.search(s)) / len(sents), 4)


def citation_validity(ans) -> float:
    """Of the citation ids the answer text actually contains, what fraction
    were in the retriever's own allowed set? (verify_citations already strips
    disallowed ids from LLM-tier answers, so this should normally be 1.0 --
    it's a direct check of that guarantee, not a duplicate of coverage.)"""
    cited_p = {int(x) for x in _CITE_P_EVAL.findall(ans.text)}
    cited_r = {int(x) for x in _CITE_R_EVAL.findall(ans.text)}
    allowed_p, allowed_r = {int(x) for x in ans.citations}, {int(x) for x in ans.review_citations}
    total = len(cited_p) + len(cited_r)
    if total == 0:
        return 1.0 if ans.missing_info else 0.0
    return float((len(cited_p & allowed_p) + len(cited_r & allowed_r)) / total)


def task_completion_proxy(query: str, ans) -> float:
    """Intent-specific, fully deterministic 0-5 task-completion score: does the
    answer contain the structures/evidence each Phase-2 capability actually
    requires? Complements the LLM judge with a reproducible, non-LLM signal --
    useful given how unreliable a small local judge has proven (see README)."""
    text = str(ans.text)
    n_p, n_r = len(set(_CITE_P_EVAL.findall(text))), len(set(_CITE_R_EVAL.findall(text)))
    if ans.missing_info:
        return 3.0 if ("اطلاعات کافی" in text or "یافت نشد" in text) else 1.0
    if ans.intent == "discovery":
        score = (2.0 if n_p >= 3 else (1.0 if n_p >= 1 else 0.0))
        score += 1.0 if "قیمت" in text else 0.0
        score += 1.0 if any(w in text for w in ("امتیاز", "توصیه", "رضایت")) else 0.0
        score += 1.0 if any(w in text for w in ("پیشنهاد", "برتر", "گزینه")) else 0.0
        return min(5.0, score)
    if ans.intent == "product_qa":
        score = 1.0 if n_p >= 1 else 0.0
        score += 2.0 if n_r >= 2 else (1.0 if n_r >= 1 else 0.0)
        score += 1.0 if any(w in text for w in ("بازبینی", "نظر")) else 0.0
        qn = pt.normalize(query)
        if any(x in qn for x in ("مشکل", "ایراد", "ضعف", "بد")):
            score += 1.0 if any(x in text for x in ("ایراد", "مشکل", "منفی", "ضعف")) else 0.0
        else:
            score += 1.0 if any(x in text for x in ("مثبت", "قوت", "توصیه", "راضی", "خوب")) else 0.0
        return min(5.0, score)
    if ans.intent == "comparison":
        score = (1.0 if n_p >= 2 else 0.0) + (1.0 if n_r >= 2 else 0.0)
        score += 0.75 if "واقعیت‌های مستقیم" in text else 0.0
        score += 0.75 if "نقاط قوت" in text else 0.0
        score += 0.75 if "نقاط ضعف" in text else 0.0
        score += 0.75 if any(w in text for w in ("جمع‌بندی", "استنباط")) else 0.0
        return min(5.0, score)
    if ans.intent == "managerial":
        score = 1.0 if any(w in text for w in ("شکایت", "نارضایتی")) else 0.0
        score += 1.0 if "برند" in text else 0.0
        score += 1.0 if "نرخ توصیه" in text else 0.0
        score += 1.0 if any(w in text for w in ("تعداد بازبینی", "تعداد محصول")) else 0.0
        score += 1.0 if (n_p >= 1 or n_r >= 1) else 0.0
        return min(5.0, score)
    return 0.0


_PROXY_STOP = {"یک", "و", "در", "از", "به", "را", "برای", "با", "می", "است",
              "این", "آن", "چه", "آیا", "کدام", "محصول", "کالا", "میخواهم", "می‌خواهم"}


def _content_tokens(text: str) -> set:
    return {t for t in pt.tokenize(text) if len(t) > 1 and t not in _PROXY_STOP}


def deterministic_quality_proxy(query: str, ans) -> dict:
    """Transparent, non-LLM proxy scores in [0, 5] -- a second, reproducible
    quality/grounding axis alongside (not instead of) the LLM judge. Explicitly
    NOT a substitute for human evaluation, just a useful control when the judge
    is unavailable or (as measured this session) unreliable on some axes."""
    q_tokens = _content_tokens(query)
    evidence_tokens = _content_tokens(_sources_text(ans))
    answer_tokens = _content_tokens(ans.text)
    relevance = min(5.0, 5.0 * (len(q_tokens & (answer_tokens | evidence_tokens)) / len(q_tokens))) if q_tokens else 0.0
    validity = citation_validity(ans)
    coverage = citation_coverage(ans.text)
    grounding = (5.0 if validity == 1.0 else 2.5) if ans.missing_info else \
        5.0 * (0.75 * validity + 0.25 * min(1.0, coverage / 0.5))
    return {"proxy_relevance_0_5": round(float(relevance), 3),
            "task_completion_proxy_0_5": round(float(task_completion_proxy(query, ans)), 3),
            "proxy_grounding_0_5": round(float(grounding), 3),
            "citation_validity": round(float(validity), 3), "citation_coverage": round(float(coverage), 3)}


def _judge_score(judge, system, user) -> int | None:
    text = judge.generate(system, user)
    if not text:
        return None
    m = re.search(r"\b([0-5])\b", text)
    return int(m.group(1)) if m else None


def _sources_text(ans) -> str:
    if ans.intent == "product_qa":
        return prompts.evidence_reviews(ans.sources)
    if ans.intent == "discovery":
        return prompts.evidence_products(ans.sources)
    return ans.text


# ---- end-to-end generative eval ----------------------------------------
def evaluate_generative(assistant, queries, judge=None) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for q in queries:
        t0 = time.time()
        a = assistant.answer(q)
        wall = round(time.time() - t0, 3)
        proxy = deterministic_quality_proxy(q, a)
        rel = faith = None
        if judge is not None and judge.available():
            rel = _judge_score(judge, prompts.JUDGE_REL_SYS,
                               prompts.JUDGE_USER.format(query=q, answer=a.text, sources=""))
            faith = _judge_score(judge, prompts.JUDGE_FAITH_SYS,
                                 prompts.JUDGE_USER.format(query=q, answer=a.text, sources=_sources_text(a)))
        rows.append({"query": q, "intent": a.intent, "tier": a.tier, "latency_s": wall,
                     "cost_usd": round(a.cost_usd, 6), "missing_info": bool(a.missing_info),
                     "citation_coverage": citation_coverage(a.text),
                     "n_citations": len(a.citations) + len(a.review_citations),
                     **proxy,
                     "relevance": rel, "faithfulness": faith})
    per_query = pd.DataFrame(rows)
    by_intent = per_query.groupby("intent").agg(
        n=("query", "count"), mean_latency_s=("latency_s", "mean"),
        mean_citation_coverage=("citation_coverage", "mean"),
        zero_citation_rate=("n_citations", lambda s: float((s == 0).mean())),
        missing_info_rate=("missing_info", "mean"),
        mean_proxy_relevance=("proxy_relevance_0_5", "mean"),
        mean_task_completion=("task_completion_proxy_0_5", "mean"),
        mean_proxy_grounding=("proxy_grounding_0_5", "mean"),
        mean_relevance=("relevance", "mean"),
        mean_faithfulness=("faithfulness", "mean")).round(3).reset_index()
    return per_query, by_intent


def build_response_eval_queries(assistant, n_contexts: int = 2):
    """A response-quality query set SEPARATE from the demo queries, built from
    high-support categories in the current sample (not human-labeled, but not
    just re-scoring the exact examples shown in the demo either). Each query's
    expected intent is checked against the router before returning -- this is a
    lightweight regression test embedded in the eval itself: if a known-intent
    query mis-routes, that's caught here rather than silently producing a
    plausible-looking but wrong-tier answer."""
    p = assistant.c.products.copy()
    if "comment_count" not in p:
        p["comment_count"] = 0
    p["comment_count"] = pd.to_numeric(p["comment_count"], errors="coerce").fillna(0).astype(int)
    p = p[p["category1_norm"].fillna("").astype(str).str.contains(r"[آ-ی]", regex=True)].copy()

    stats = (p.groupby("category1_norm").agg(n_products=("product_id", "nunique"),
                                              n_reviews=("comment_count", "sum")).reset_index())
    stats = stats[(stats["n_products"] >= 8) & (stats["n_reviews"] >= 80)].sort_values(
        ["n_reviews", "n_products"], ascending=False)

    queries, expected, contexts = [], [], []
    for row in stats.itertuples(index=False):
        cat = pt.normalize(row.category1_norm)
        if not cat:
            continue
        g = p[p["category1_norm"] == row.category1_norm]
        pair = (g[g["comment_count"] >= 3].sort_values(
            ["comment_count", "rate_count", "product_rate_clean"], ascending=False)
            ["product_id"].head(2).astype(int).tolist())
        if len(pair) < 2:
            continue
        queries += [f"در دستهٔ {cat} چند محصول با رضایت خوب کاربران معرفی کن",
                   f"آیا محصول {pair[0]} از نظر کیفیت و تجربهٔ کاربران ارزش خرید دارد؟",
                   f"مهم‌ترین ایرادها و نقاط ضعف محصول {pair[1]} از نظر کاربران چیست؟",
                   f"محصول {pair[0]} و محصول {pair[1]} را از نظر قیمت، رضایت کاربران و نقاط قوت و ضعف مقایسه کن",
                   f"در دستهٔ {cat} شکایت‌های پرتکرار چیست و کدام محصولات با نظر کافی نرخ توصیهٔ پایین‌تری دارند؟"]
        expected += ["discovery", "product_qa", "product_qa", "comparison", "managerial"]
        contexts.append({"category": cat, "product_ids": pair,
                         "n_products": int(row.n_products), "n_reviews": int(row.n_reviews)})
        if len(contexts) >= n_contexts:
            break

    if not contexts:
        queries, expected, contexts = default_queries(assistant), None, [{"fallback_to_demo": True}]

    routing = [{"query": q, "expected": exp, "actual": assistant.router.route(q).intent,
               "passed": assistant.router.route(q).intent == exp}
              for q, exp in zip(queries, expected or [])]
    failures = [r for r in routing if not r["passed"]]
    if failures:
        log.warning("response-eval routing regression: %s", failures)

    meta = {"source": ("held_out_programmatic_multi_category_not_human_labeled"
                       if not contexts[0].get("fallback_to_demo") else "demo_fallback"),
           "n_queries": len(queries), "n_contexts": len(contexts), "contexts": contexts,
           "routing_checks_passed": not failures, "routing": routing}
    return queries, meta


def failure_analysis(assistant, judge=None, n_retrieval: int = 40, k: int = 10) -> dict:
    """Collect concrete failure examples across the system, per the brief's
    'Failure Analysis' requirement: retrieval misses, and generation failures
    (missing-info, zero-citation, or low judge faithfulness), with likely causes.
    """
    products = assistant.c.products
    # --- retrieval misses: the product's title should retrieve its own id ---
    pids = (products.sort_values("comment_count", ascending=False)["product_id"]
            .head(n_retrieval).astype(int).tolist())
    retrieval_failures = []
    for pid in pids:
        title = assistant.c.product(pid)["title_fa"]
        hits = [h["product_id"] for h in assistant.pidx.search(title, k=k)]
        if pid not in hits:
            retrieval_failures.append({
                "query": title, "gold_product_id": pid, "top_returned": hits[:3],
                "cause": "generic/short title or many near-duplicate listings dilute the match"})

    # --- generation failures on probe queries (some designed to fail) ---
    two = pids[:2] if len(pids) >= 2 else pids
    probes = [
        "یک محصول کاملاً بی‌ربط و ناموجود مثلا سفینه فضایی مریخ‌نورد می‌خواهم",  # out-of-catalog
        "نظر کاربران درباره محصول 999999999 چیست؟",                              # non-existent id
        "این دو تا رو مقایسه کن",                                               # comparison w/o ids
        f"آیا محصول {two[0]} ارزش خرید دارد؟" if two else "آیا ارزش خرید دارد؟",
    ]
    gen_failures = []
    for q in probes:
        a = assistant.answer(q)
        n_cit = len(a.citations) + len(a.review_citations)
        faith = None
        if judge is not None and judge.available():
            faith = _judge_score(judge, prompts.JUDGE_FAITH_SYS,
                                 prompts.JUDGE_USER.format(query=q, answer=a.text, sources=_sources_text(a)))
        reasons = []
        if a.missing_info:
            reasons.append("missing_info: no grounded evidence matched the query")
        if n_cit == 0:
            reasons.append("zero citations: nothing to ground an answer on")
        if a.needs_clarification:
            reasons.append("needs clarification: under-specified request")
        if faith is not None and faith <= 2:
            reasons.append(f"low faithfulness ({faith}/5) from the judge")
        if reasons:
            gen_failures.append({"query": q, "intent": a.intent, "tier": a.tier,
                                 "answer": a.text[:200], "reasons": reasons})
    return {
        "retrieval": {"n_checked": len(pids), "n_failed": len(retrieval_failures),
                      "examples": retrieval_failures[:5]},
        "generation": {"n_probes": len(probes), "n_failed": len(gen_failures),
                       "examples": gen_failures},
        "mitigations": [
            "extractive fallback guarantees a grounded ($0) answer when the LLM is unsure",
            "verify_citations strips any id the retriever did not return (no fabricated refs)",
            "the router asks for clarification instead of guessing on under-specified comparisons",
        ],
    }


def human_eval_query_set(assistant) -> list[str]:
    """A fixed, diverse ~16-query set for human-vs-judge evaluation: the 6 default
    queries (one per intent-ish scenario) plus harder/adversarial probes so the
    comparison isn't only on easy cases."""
    base = default_queries(assistant)
    pids = (assistant.c.products.sort_values("comment_count", ascending=False)["product_id"]
            .head(4).astype(int).tolist())
    extra = [
        "یک محصول کاملاً بی‌ربط و ناموجود مثلا سفینه فضایی مریخ‌نورد می‌خواهم",
        "نظر کاربران درباره محصول 999999999 چیست؟",
        "این دو تا رو مقایسه کن",
        f"بهترین قیمت برای محصول {pids[2] if len(pids) > 2 else pids[0]} چقدره؟",
        f"آیا محصول {pids[-1]} ارزش خرید دارد؟",
        "چه گوشی موبایلی زیر ۱۰ میلیون تومان با باتری خوب پیشنهاد میدی؟",
        f"محصول {pids[0]} چه ایرادهایی داره؟",
        "کیفیت ساخت این محصولات چطوره و کاربرا راضی بودن؟",
        f"محصول {pids[1]} و محصول {pids[2] if len(pids) > 2 else pids[0]} کدوم بهتره؟",
        "پرتکرارترین مشکلات کاربران در این دسته از محصولات چیه؟",
    ]
    seen, out = set(), []
    for q in base + extra:
        if q not in seen:
            seen.add(q)
            out.append(q)
    return out[:16]


def _answer_hash(text: str) -> str:
    import hashlib
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]


def build_human_eval_candidates(assistant, judge=None) -> list[dict]:
    """Generate the candidate set for human labeling: query, the system's answer,
    the evidence it was grounded on, and (if a judge is configured) the judge's
    own relevance/faithfulness scores for the same items -- so the human labels
    can be directly compared without re-running the judge later.

    Each row carries an `answer_hash` (of the exact answer text). A label is
    only ever compared against the judge score for the SAME answer text it was
    actually written for -- if the assistant's logic changes and the answer to
    the same query changes, the old label is now scoring a different, no
    longer existing response. Comparing it to a fresh judge score anyway would
    silently conflate "the judge disagrees with a human" with "the system's
    answer changed since this was labeled," which is a real methodological
    trap this project hit and is guarding against here, not hypothetically."""
    rows = []
    for q in human_eval_query_set(assistant):
        a = assistant.answer(q)
        rel = faith = None
        if judge is not None and judge.available():
            rel = _judge_score(judge, prompts.JUDGE_REL_SYS,
                               prompts.JUDGE_USER.format(query=q, answer=a.text, sources=""))
            faith = _judge_score(judge, prompts.JUDGE_FAITH_SYS,
                                 prompts.JUDGE_USER.format(query=q, answer=a.text, sources=_sources_text(a)))
        rows.append({"query": q, "intent": a.intent, "answer": a.text, "answer_hash": _answer_hash(a.text),
                     "evidence": _sources_text(a)[:1500], "judge_relevance": rel, "judge_faithfulness": faith})
    return rows


def human_eval_comparison(candidates: list[dict], labels: dict) -> dict:
    """Compare a human's 0-5 relevance/faithfulness labels (keyed by query) against
    the judge scores already attached to `candidates`. `labels` format:
    {query: {"relevance": int, "faithfulness": int, "answer_hash": str}}.
    A label without a matching `answer_hash` (or without one at all, e.g. an
    older label file predating this check) is treated as STALE -- scored
    against an answer the system no longer produces -- and excluded from
    agreement, counted separately in `n_stale` rather than silently mixed in."""
    rel_h, rel_j, faith_h, faith_j = [], [], [], []
    matched, stale = [], []
    for c in candidates:
        lab = labels.get(c["query"])
        if not lab:
            continue
        if lab.get("answer_hash") != c.get("answer_hash"):
            stale.append(c["query"])           # missing hash (older label file) also counts as stale
            continue
        matched.append(c["query"])
        if c.get("judge_relevance") is not None and lab.get("relevance") is not None:
            rel_h.append(lab["relevance"]); rel_j.append(c["judge_relevance"])
        if c.get("judge_faithfulness") is not None and lab.get("faithfulness") is not None:
            faith_h.append(lab["faithfulness"]); faith_j.append(c["judge_faithfulness"])
    return {"n_labeled": len(matched), "n_stale": len(stale), "queries_labeled": matched,
            "stale_queries": stale,
            "relevance_agreement": judge_human_agreement(rel_h, rel_j),
            "faithfulness_agreement": judge_human_agreement(faith_h, faith_j)}


def judge_human_agreement(human: list[float], judge: list[float]) -> dict:
    """Spearman correlation between judge scores and a small hand-labeled set."""
    if len(human) < 3 or len(human) != len(judge):
        return {}
    try:
        from scipy.stats import spearmanr
        rho, p = spearmanr(human, judge)
        return {"spearman_rho": round(float(rho), 4), "p_value": round(float(p), 4), "n": len(human)}
    except Exception:
        r = float(np.corrcoef(human, judge)[0, 1])
        return {"pearson_r": round(r, 4), "n": len(human)}


# ---- default eval query set --------------------------------------------
def default_queries(assistant) -> list[str]:
    pids = (assistant.c.products.sort_values("comment_count", ascending=False)["product_id"]
            .head(2).astype(int).tolist())
    cat = assistant.c.products["category1_norm"].replace("نامشخص", np.nan).dropna().value_counts().index[0]
    return [
        "یک محصول اقتصادی و باکیفیت برای استفاده روزمره پیشنهاد بده",
        "یک کالای مناسب زیر ۵۰۰ هزار تومان می‌خواهم",
        f"آیا محصول {pids[0]} کیفیت خوبی دارد و کاربران راضی بودند؟",
        f"مشکلات و ایرادهای محصول {pids[1]} چیست؟",
        f"محصول {pids[0]} و محصول {pids[1]} را از نظر کیفیت مقایسه کن",
        f"پرتکرارترین شکایت‌ها و نقاط ضعف در دستهٔ {cat} چیست؟",
    ]


def run(n_retrieval: int = 30) -> dict:
    """Run the full Phase-4 suite and write artifacts/metrics/phase4_metrics.*"""
    budget = BudgetTracker()
    assistant = build_assistant()
    judge = judge_llm(budget=budget)
    log.info("assistant backend=%s | judge mode=%s", assistant.llm.backend, config.JUDGE_MODE)

    retrieval = evaluate_retrieval(assistant, n_queries=n_retrieval)
    log.info("retrieval: %s", retrieval)

    ablation_r = retrieval_ablation(assistant, n_queries=n_retrieval)
    log.info("retrieval ablation: %s", ablation_r["by_method"])

    natural = evaluate_retrieval_natural(assistant, n_queries=n_retrieval)
    log.info("natural-language retrieval benchmark: hybrid=%s lexical=%s verdict=%s",
             natural.get("hybrid"), natural.get("lexical_baseline"), natural.get("quality_verdict_by_ndcg"))

    queries, response_eval_meta = build_response_eval_queries(assistant, n_contexts=2)
    per_query, by_intent = evaluate_generative(assistant, queries, judge=judge)

    # baseline control
    baseline = LexicalBaseline(assistant.c)
    base_hits = baseline.discover("یک کالای ارزان و باکیفیت", k=5)

    # failure analysis (concrete examples + likely causes)
    failures = failure_analysis(assistant, judge=judge)
    log.info("failures: retrieval %d/%d, generation %d/%d",
             failures["retrieval"]["n_failed"], failures["retrieval"]["n_checked"],
             failures["generation"]["n_failed"], failures["generation"]["n_probes"])

    # human-vs-judge eval: generate/refresh the candidate set every run (so it's
    # always current), then compare against hand labels if the user has scored them
    candidates = build_human_eval_candidates(assistant, judge=judge)
    (config.METRICS_DIR / "human_eval_candidates.json").write_text(
        json.dumps(candidates, ensure_ascii=False, indent=2), encoding="utf-8")
    labels_file = config.METRICS_DIR / "human_eval_labels.json"
    if labels_file.exists():
        labels = json.loads(labels_file.read_text(encoding="utf-8"))
        human_eval = human_eval_comparison(candidates, labels)
    else:
        human_eval = {"n_labeled": 0, "note": "no human_eval_labels.json yet -- "
                      "see artifacts/metrics/human_eval_candidates.json to label it"}
    log.info("human-eval agreement: %s", human_eval)

    # Phase-3 macro-F1 + leakage ablation (if trained)
    p3 = {}
    p3_file = config.METRICS_DIR / "phase3_metrics.json"
    if p3_file.exists():
        p3 = json.loads(p3_file.read_text(encoding="utf-8"))

    metrics = {
        "run_mode": config.RUN_MODE, "judge_mode": config.JUDGE_MODE,
        "retrieval_quality": retrieval,
        "retrieval_ablation": ablation_r,
        "retrieval_quality_natural": natural,
        "generation": {
            "n_queries": int(len(per_query)),
            "evaluation_set": response_eval_meta,
            "mean_latency_s": round(float(per_query["latency_s"].mean()), 3),
            "total_cost_usd": round(float(per_query["cost_usd"].sum()), 6),
            "mean_citation_coverage": round(float(per_query["citation_coverage"].mean()), 3),
            "mean_proxy_relevance_0_5": round(float(per_query["proxy_relevance_0_5"].mean()), 3),
            "mean_task_completion_proxy_0_5": round(float(per_query["task_completion_proxy_0_5"].mean()), 3),
            "mean_proxy_grounding_0_5": round(float(per_query["proxy_grounding_0_5"].mean()), 3),
            "mean_citation_validity": round(float(per_query["citation_validity"].mean()), 3),
            "mean_relevance": (None if per_query["relevance"].isna().all()
                               else round(float(per_query["relevance"].mean()), 3)),
            "mean_faithfulness": (None if per_query["faithfulness"].isna().all()
                                  else round(float(per_query["faithfulness"].mean()), 3)),
            "by_intent": by_intent.to_dict("records"),
        },
        "prediction_macro_f1": p3.get("test_macro_f1"),
        "prediction_grouped_macro_f1": p3.get("grouped_macro_f1"),
        "prediction_primary_macro_f1": p3.get("primary_macro_f1", p3.get("grouped_macro_f1")),
        "prediction_naive_split_product_overlap_pct": p3.get("naive_split_product_overlap_pct"),
        "prediction_leakage_ablation": p3.get("leakage_ablation"),
        "failure_analysis": failures,
        "human_eval_agreement": human_eval,
        "cost": budget.summary(),
        "baseline_control": {"query": "یک کالای ارزان و باکیفیت",
                             "top_product_ids": [int(x) for x in base_hits["product_id"].tolist()]},
    }
    (config.METRICS_DIR / "phase4_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    per_query.to_csv(config.METRICS_DIR / "phase4_per_query.csv", index=False, encoding="utf-8-sig")
    by_intent.to_csv(config.METRICS_DIR / "phase4_by_intent.csv", index=False, encoding="utf-8-sig")
    log.info("wrote phase4 metrics to %s", config.METRICS_DIR)
    return metrics

In [20]:
# retrieval quality (auto-labeled: product title -> its own id)
retr = evaluate_retrieval(assistant, n_queries=20, k=10); print("retrieval:", retr)

# retrieval ablation (bonus): hybrid vs. dense-only vs. BM25-only on the same
# auto-labeled queries. Caveat: title-as-query is a near-exact lexical match, so
# this benchmark structurally favors BM25 -- see the package README for the
# full discussion of why hybrid can (and does, on the full corpus) lose to
# BM25-only on THIS specific benchmark without that meaning hybrid is worse in general.
retr_ablation = retrieval_ablation(assistant, n_queries=20, k=10)
print("retrieval ablation (by method):", retr_ablation["by_method"])
print("hybrid vs best single-method MRR lift:", retr_ablation["hybrid_vs_best_single_mrr_lift"])

# natural-language retrieval benchmark (brand+category+partial-title paraphrase
# queries, NOT exact titles) -- the fair test of hybrid's value, since the
# title-exact-match benchmark above structurally favors BM25. On the full
# corpus this benchmark independently confirms the same honest finding: the
# lexical/BM25 baseline currently beats hybrid on nDCG for this catalogue with
# this embedding model -- reported as measured, not spun into a claimed win.
natural = evaluate_retrieval_natural(assistant, n_queries=20, k=10)
print("natural-language benchmark -- hybrid:", natural.get("hybrid"),
      "| lexical baseline:", natural.get("lexical_baseline"),
      "| verdict:", natural.get("quality_verdict_by_ndcg"))

# generative eval on a response-quality query set SEPARATE from the demo
# queries (built from held-out categories); routing is checked against each
# query's expected intent as a lightweight regression test embedded in eval.
judge = judge_llm() if JUDGE_MODE != "none" else None
queries, response_eval_meta = build_response_eval_queries(assistant, n_contexts=2)
print("routing_checks_passed:", response_eval_meta["routing_checks_passed"])
per_query, by_intent = evaluate_generative(assistant, queries, judge=judge)
display(per_query); display(by_intent)

retrieval: {'n_queries': 20, 'k': 10, 'recall@k': np.float64(0.9), 'mrr': np.float64(0.9), 'ndcg@k': np.float64(0.9)}


retrieval ablation (by method): {'dense': {'recall@k': np.float64(0.75), 'mrr': np.float64(0.6479), 'ndcg@k': np.float64(0.6723)}, 'bm25': {'recall@k': np.float64(0.95), 'mrr': np.float64(0.95), 'ndcg@k': np.float64(0.95)}, 'hybrid': {'recall@k': np.float64(0.9), 'mrr': np.float64(0.9), 'ndcg@k': np.float64(0.9)}}
hybrid vs best single-method MRR lift: -0.05


natural-language benchmark -- hybrid: {'recall@k': 0.85, 'mrr': 0.485, 'ndcg@k': 0.5749} | lexical baseline: {'recall@k': 0.85, 'mrr': 0.7042, 'ndcg@k': 0.7381} | verdict: lexical_baseline_better


routing_checks_passed: False


,query,intent,tier,latency_s,cost_usd,missing_info,citation_coverage,n_citations,proxy_relevance_0_5,task_completion_proxy_0_5,proxy_grounding_0_5,citation_validity,relevance,faithfulness
0,در دستهٔ اسباب بازی چند محصول با رضایت خوب کار...,product_qa,extractive,0.041,0.0,False,0.750,2,1.667,4.0,5.000,1.0,None,None
1,آیا محصول 1516000 از نظر کیفیت و تجربهٔ کاربرا...,product_qa,extractive,0.038,0.0,False,0.714,4,1.875,5.0,5.000,1.0,None,None
2,مهم‌ترین ایرادها و نقاط ضعف محصول 1025606 از ن...,product_qa,extractive,0.041,0.0,False,0.800,3,1.667,5.0,5.000,1.0,None,None
3,محصول 1516000 و محصول 1025606 را از نظر قیمت، ...,comparison,extractive,0.079,0.0,False,0.800,9,3.636,5.0,5.000,1.0,None,None
4,در دستهٔ اسباب بازی شکایت‌های پرتکرار چیست و ک...,managerial,extractive,4.168,0.0,False,0.372,16,3.000,5.0,4.680,1.0,None,None
5,در دستهٔ مراقبت پوست چند محصول با رضایت خوب کا...,discovery,extractive,0.133,0.0,False,0.242,8,2.222,5.0,4.356,1.0,None,None
6,آیا محصول 442022 از نظر کیفیت و تجربهٔ کاربران...,product_qa,extractive,0.048,0.0,False,0.667,3,1.875,5.0,5.000,1.0,None,None
7,مهم‌ترین ایرادها و نقاط ضعف محصول 81739 از نظر...,discovery,extractive,0.037,0.0,False,0.296,8,2.222,5.0,4.491,1.0,None,None
8,محصول 442022 و محصول 81739 را از نظر قیمت، رضا...,comparison,extractive,0.007,0.0,False,0.000,0,0.455,0.0,0.000,0.0,None,None
9,در دستهٔ مراقبت پوست شکایت‌های پرتکرار چیست و ...,managerial,extractive,4.024,0.0,False,0.296,16,3.000,5.0,4.491,1.0,None,None


,intent,n,mean_latency_s,mean_citation_coverage,zero_citation_rate,missing_info_rate,mean_proxy_relevance,mean_task_completion,mean_proxy_grounding,mean_relevance,mean_faithfulness
0,comparison,2,0.043,0.400,0.5,0.0,2.046,2.50,2.500,NaN,NaN
1,discovery,2,0.085,0.269,0.0,0.0,2.222,5.00,4.424,NaN,NaN
2,managerial,2,4.096,0.334,0.0,0.0,3.000,5.00,4.586,NaN,NaN
3,product_qa,4,0.042,0.733,0.0,0.0,1.771,4.75,5.000,NaN,NaN


In [21]:
# baseline control (no embeddings, no LLM)
display(LexicalBaseline(catalog).discover("یک کالای ارزان و باکیفیت", k=5))

,product_id,title_fa,price_clean,product_rate_clean
14784,173583,کتاب راند، از دفترچه خاطرات یک دانشجوی پزشکی ا...,262800,100
15846,4230451,کتاب آوازی برای یک نهنگ اثر لین کلی انتشارات پ...,1392000,94
15863,3598889,کتاب خاطرات یک بچه ی چلمن اثر جف کینی انتشارات...,1162900,94
16117,11416119,کتاب یک قدم تا بهشت اثر آن نا انتشارات نگاه آشنا,536600,94
16017,5702430,کتاب حکایت های بهلول 4 مردی که یک گله مرغ و خر...,147000,94


### 4.0 Failure analysis
Concrete failure cases + likely causes (same `failure_analysis` the package and dashboard use).

In [22]:
fa = failure_analysis(assistant, judge=judge)
print("retrieval misses:", fa["retrieval"]["n_failed"], "/", fa["retrieval"]["n_checked"])
print("generation failures on probes:", fa["generation"]["n_failed"], "/", fa["generation"]["n_probes"])
for ex in fa["generation"]["examples"]:
    print(" -", ex["intent"], "|", ex["query"][:55], "->", ex["reasons"])

retrieval misses: 2 / 40
generation failures on probes: 0 / 4


### 4.0.1 Human-vs-judge evaluation (bonus, full-data artifact)

`build_human_eval_candidates` + `human_eval_comparison` (defined above) generate a
fixed 16-query set with judge scores and compare it against hand labels. This is run
on the **full corpus** by `python run.py eval` (not re-run on this notebook's small
sample, since a meaningful human-labeled set needs the real deployed assistant's
answers) — see `artifacts/metrics/human_eval_candidates.json` /
`human_eval_labels.json` / `phase4_metrics.json["human_eval_agreement"]` in the
package repo, and the README's "LLM-as-judge" section for the measured result
(faithfulness ρ≈0.52, relevance unparseable on 16/16 — a real limitation of the
1.5B local judge, not hidden).

## 4.1 No notebook-vs-application discrepancy

This notebook and the packaged application call the **same functions on the same
deterministic sample**, so their substantive outputs match exactly. The summary
below is byte-for-byte identical to `python run.py demo --sample-size 20000` (retrieval
metrics, Macro-F1, and the demo citations are deterministic; only wall-clock latency
varies).

In [23]:
import json
top = catalog.products.sort_values("comment_count", ascending=False)["product_id"].head(2).astype(int).tolist()
demos = {}
for name, q in [("discovery","یک کالای اقتصادی و باکیفیت زیر ۵۰۰ هزار تومان می‌خواهم"),
                ("product_qa", f"آیا کاربران از کیفیت محصول {top[0]} راضی بودند؟"),
                ("comparison", f"محصول {top[0]} و محصول {top[1]} را مقایسه کن")]:
    a = assistant.answer(q)
    demos[name] = {"intent": a.intent, "tier": a.tier,
                   "citations": sorted(a.citations), "review_citations": sorted(a.review_citations)}
summary = {"sample_size": SAMPLE_SIZE, "seed": 42,
           "n_products": int(len(products_df)), "n_comments": int(len(comments_df)),
           "retrieval_quality": retr,
           "prediction_test_macro_f1": p3_metrics["test_macro_f1"],
           "prediction_grouped_macro_f1": p3_metrics["grouped_macro_f1"],
           "prediction_primary_macro_f1": p3_metrics["primary_macro_f1"],
           "prediction_naive_split_product_overlap_pct": p3_metrics["naive_split_product_overlap_pct"],
           "prediction_leakage_ablation": p3_metrics["leakage_ablation"],
           "demos": demos}
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "sample_size": 20000,
  "seed": 42,
  "n_products": 17043,
  "n_comments": 20000,
  "retrieval_quality": {
    "n_queries": 20,
    "k": 10,
    "recall@k": 0.9,
    "mrr": 0.9,
    "ndcg@k": 0.9
  },
  "prediction_test_macro_f1": 0.6103,
  "prediction_grouped_macro_f1": 0.4924,
  "prediction_primary_macro_f1": 0.4924,
  "prediction_naive_split_product_overlap_pct": 14.34,
  "prediction_leakage_ablation": {
    "text_only_macro_f1": 0.6103,
    "text_plus_numeric_macro_f1": 0.6626,
    "leakage_lift": 0.0523,
    "note": "The lift from adding rate_clean/likes/is_buyer is leakage: rate restates the label, likes are post-hoc. The final model excludes them."
  },
  "demos": {
    "discovery": {
      "intent": "discovery",
      "tier": "extractive",
      "citations": [
        170694,
        176113,
        354569,
        822241,
        1021554,
        3135078,
        4279784,
        7523408
      ],
      "review_citations": []
    },
    "product_qa": {
      "intent": "prod

## 5 · Summary

- **Phase 1** cleans the raw data into a documented schema and profiles it with Plotly.
- **Phase 2** answers four kinds of Persian request with hybrid retrieval and a
  **citation contract** — every claim is backed by a retrieved `[محصول id]` /
  `[بازبینی id]`, and the extractive tier guarantees grounding at **$0**.
- **Phase 3** predicts `recommendation_status`; the majority baseline shows the model
  is a real improvement, a product-grouped split rules out product-level leakage
  (reported as the **primary** metric, alongside the naive split's measured product
  overlap), and a text-vs-text+numeric ablation quantifies the rate/likes leak we
  deliberately excluded. Metric: **Macro-F1**.
- **Phase 4** measures retrieval (recall/MRR/nDCG, plus a hybrid-vs-single-method
  ablation), grounding, latency, cost and per-intent failures, with a non-LLM
  baseline as control.

**Bonus work** (full detail + measured numbers in the package README and the
dashboard's "🏆 Bonus & Engineering" page): a deterministic $0 intent router;
measured caching/algorithmic optimizations (BM25 rewrite, fast tokenizer, mmap'd
vectors, etc.); the retrieval ablation above; a human-vs-judge evaluation
comparison; and a LoRA fine-tune of a Persian encoder (ParsBERT) compared against
the TF-IDF baseline on the identical product-grouped split (`python run.py lora`).

Run the production package with `python run.py all` (full data), `python run.py lora`
(LoRA bonus), or `python run.py dashboard` for the interactive UI.